# Branches/Transport of the subtropical cells

### Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.collections as mcollections
import matplotlib.colors as colors
import numpy as np
import cmocean
import processing_utils as proc_utils
import cesm2_lens_utils
import analysis_funcs as afuncs
import xesmf as xe
import pop_tools
import cftime
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.stats import gaussian_kde

In [ ]:
len(region_idx_f)

In [ ]:
FOSI_pd_obs_array = np.load('/glade/derecho/scratch/cassiacai/FOSI_SH_traj_PD_fullarray.npy')


In [ ]:
# For FOSI
print(lon_trunc_f.dims)
print(lon_trunc_f.shape)  # expecting (trajectory, time)?

# How do you define "reaches the equator"?
# Is it particles that cross lat=0 at any point?
print(lat_trunc_f.min())  # do any go to 0 or above?

# What does region_idx_f contain?
print(region_idx_f)
print(len(region_idx_f))

# Check lat range of SH particles in FOSI
sh_lats_f = lat_trunc_f.isel(trajectory=region_idx_f)
print(f"SH particle lat range: {sh_lats_f.min().values:.2f} to {sh_lats_f.max().values:.2f}")

In [ ]:
# FOSI: which of the 60 SH particles ever reach lat >= -2 (2°S)
sh_lats_f = lat_trunc_f.isel(trajectory=region_idx_f)  # shape (60, 244)
reaches_equator_f = (sh_lats_f >= -2).any(dim='obs')    # True/False for each particle
frac_fosi = reaches_equator_f.sum().values / 60

print(f"FOSI: {reaches_equator_f.sum().values}/60 particles reach 2°S ({frac_fosi*100:.1f}%)")

In [ ]:
# For FOSI — where do particles actually cross 2°S?
sh_lats_f = lat_trunc_f.isel(trajectory=region_idx_f)  # (60, 244)
sh_lons_f = lon_trunc_f.isel(trajectory=region_idx_f)  # (60, 244)

# For each particle, find the longitude when it first crosses -2
for i in range(60):
    lat_traj = sh_lats_f.isel(trajectory=i).values
    lon_traj = sh_lons_f.isel(trajectory=i).values
    cross = np.where(lat_traj >= -2)[0]
    if len(cross) > 0:
        first = cross[0]
        print(f"Particle {i}: crosses 2°S at lon={lon_traj[first]:.1f}")

In [ ]:
# No longitude filter needed — all crossings are Pacific
reaches_equator_f = (sh_lats_f >= -2).any(dim='obs')
frac_fosi = reaches_equator_f.sum().values / 60
print(f"FOSI: {reaches_equator_f.sum().values}/60 reach 2°S ({frac_fosi*100:.1f}%)")

# Also report the two pathways
for i in range(60):
    lat_traj = sh_lats_f.isel(trajectory=i).values
    lon_traj = sh_lons_f.isel(trajectory=i).values
    cross = np.where(lat_traj >=0)[0]
    if len(cross) > 0:
        first = cross[0]
        if lon_traj[first] <= 165:
            pathway = 'WBC'
        else:
            pathway = 'Interior'
        print(f"Particle {i}: {pathway} (lon={lon_traj[first]:.1f})")

In [ ]:
# Recalculate FOSI pathway breakdown correctly
sh_lats_f = lat_trunc_f.isel(trajectory=region_idx_f)
sh_lons_f = lon_trunc_f.isel(trajectory=region_idx_f)

wbc_count_f = 0
interior_count_f = 0

for i in range(60):
    lat_traj = sh_lats_f.isel(trajectory=i).values
    lon_traj = sh_lons_f.isel(trajectory=i).values
    cross = np.where(lat_traj >= -2)[0]
    if len(cross) > 0:
        first = cross[0]
        if lon_traj[first] <= 165:
            wbc_count_f += 1
        else:
            interior_count_f += 1

print(f"FOSI total:    {(wbc_count_f + interior_count_f)}/60 = {(wbc_count_f + interior_count_f)/60*100:.1f}%")
print(f"FOSI WBC:      {wbc_count_f}/60 = {wbc_count_f/60*100:.1f}%")
print(f"FOSI Interior: {interior_count_f}/60 = {interior_count_f/60*100:.1f}%")
print(f"Check sum:     {(wbc_count_f + interior_count_f)/60*100:.1f}% (should match total)")

print(f"\nLENS total:    {fracs_lens.mean()*100:.1f}% ± {fracs_lens.std()*100:.1f}%")
print(f"LENS WBC:      {wbc_lens.mean()*100:.1f}% ± {wbc_lens.std()*100:.1f}%")
print(f"LENS Interior: {interior_lens.mean()*100:.1f}% ± {interior_lens.std()*100:.1f}%")

In [ ]:
# ============================================================
# FOSI
# ============================================================
sh_lats_f = lat_trunc_f.isel(trajectory=region_idx_f)
sh_lons_f = lon_trunc_f.isel(trajectory=region_idx_f)

wbc_count_f = 0
interior_count_f = 0

for i in range(60):
    lat_traj = sh_lats_f.isel(trajectory=i).values
    lon_traj = sh_lons_f.isel(trajectory=i).values
    cross = np.where(lat_traj >= -2)[0]
    if len(cross) > 0:
        first = cross[0]
        if lon_traj[first] <= 165:
            wbc_count_f += 1
        else:
            interior_count_f += 1

frac_fosi = (wbc_count_f + interior_count_f) / 60
print(f"FOSI total:    {wbc_count_f + interior_count_f}/60 = {frac_fosi*100:.1f}%")
print(f"FOSI WBC:      {wbc_count_f}/60 = {wbc_count_f/60*100:.1f}%")
print(f"FOSI Interior: {interior_count_f}/60 = {interior_count_f/60*100:.1f}%")

# ============================================================
# LENS
# ============================================================
fracs_lens = []
wbc_lens = []
interior_lens = []

for i in range(len(ENS_MEMBERS_LIST)):
    lat_l = lat_trunc_list[i]
    lon_l = lon_trunc_list[i]
    region_idx_l = region_indx_list[i]
    n_particles = len(region_idx_l)

    sh_lats_l = lat_l.isel(trajectory=region_idx_l)
    sh_lons_l = lon_l.isel(trajectory=region_idx_l)

    reaches = (sh_lats_l >= -2).any(dim='obs')
    frac = reaches.sum().values / n_particles
    fracs_lens.append(frac)

    wbc_count = 0
    interior_count = 0
    for j in range(n_particles):
        lat_traj = sh_lats_l.isel(trajectory=j).values
        lon_traj = sh_lons_l.isel(trajectory=j).values
        cross = np.where(lat_traj >= -2)[0]
        if len(cross) > 0:
            first = cross[0]
            if lon_traj[first] <= 165:
                wbc_count += 1
            else:
                interior_count += 1

    wbc_lens.append(wbc_count / n_particles)
    interior_lens.append(interior_count / n_particles)
    print(f"Member {ENS_MEMBERS_LIST[i]:3d}: {reaches.sum().values:2d}/{n_particles} ({frac*100:.1f}%) — WBC: {wbc_count} ({wbc_count/n_particles*100:.1f}%), Interior: {interior_count} ({interior_count/n_particles*100:.1f}%)")

fracs_lens = np.array(fracs_lens)
wbc_lens = np.array(wbc_lens)
interior_lens = np.array(interior_lens)

# ============================================================
# SUMMARY
# ============================================================
print(f"\n{'='*60}")
print(f"FOSI total:    {frac_fosi*100:.1f}%")
print(f"LENS total:    {fracs_lens.mean()*100:.1f}% ± {fracs_lens.std()*100:.1f}%")
print(f"\nFOSI WBC:      {wbc_count_f/60*100:.1f}%")
print(f"LENS WBC:      {wbc_lens.mean()*100:.1f}% ± {wbc_lens.std()*100:.1f}%")
print(f"\nFOSI Interior: {interior_count_f/60*100:.1f}%")
print(f"LENS Interior: {interior_lens.mean()*100:.1f}% ± {interior_lens.std()*100:.1f}%")

In [ ]:
def get_region_lonlat_raw(lat_data, lon_data, lat_min, lat_max, lon_min, lon_max):
    """
    Get indices of particles whose INITIAL position (obs=0) falls within the given region.
    Works on raw (non-truncated) xarray DataArrays.
    """
    # Use initial position only (obs index 0)
    lat_init = lat_data.isel(obs=0)
    lon_init = lon_data.isel(obs=0)
    
    in_region_mask = (
        (lat_init >= lat_min) & 
        (lat_init <= lat_max) &
        (lon_init >= lon_min) & 
        (lon_init <= lon_max)
    )
    
    region_indices = np.where(in_region_mask.values)[0]
    print(f"Found {len(region_indices)} particles in region")
    return region_indices
    
# FOSI — no truncation
l_time_f, lon_f_raw, lat_f_raw, z_f_raw = compute_data(fosi_pt)

# Use the raw lon/lat directly instead of lon_trunc_f/lat_trunc_f
region_idx_f_raw = get_region_lonlat_raw(
    lat_f_raw, lon_f_raw,
    lat_min=-40, lat_max=-30,
    lon_min=215, lon_max=275)

sh_lats_f_raw = lat_f_raw.isel(trajectory=region_idx_f_raw)
sh_lons_f_raw = lon_f_raw.isel(trajectory=region_idx_f_raw)

# Now check with equator at 0°
reaches_0 = (sh_lats_f_raw >= 0).any(dim='obs')
reaches_2S = (sh_lats_f_raw >= -2).any(dim='obs')

print(f"FOSI without truncation:")
print(f"  Reaching 0°:  {reaches_0.sum().values}/60 = {reaches_0.sum().values/60*100:.1f}%")
print(f"  Reaching 2°S: {reaches_2S.sum().values}/60 = {reaches_2S.sum().values/60*100:.1f}%")

In [ ]:
fracs_lens_raw = []

for i, ENS_MEMB in enumerate(ENS_MEMBERS_LIST):
    file_path = '/glade/derecho/scratch/cassiacai/particle_trajectories/particle_trajectories_lens{}_SH_startingat{}m_1958_1977.zarr'.format(ENS_MEMB, INIT_DEPTH)
    lens_pt = xr.open_zarr(file_path)
    l_time, l_lon, l_lat, l_z = compute_data(lens_pt)
    
    region_idx_raw = get_region_lonlat(l_lat, l_lon, -40, -30, 215, 275)
    
    sh_lats = l_lat.isel(trajectory=region_idx_raw)
    
    reaches_0  = (sh_lats >= 0).any(dim='obs')
    reaches_2S = (sh_lats >= -2).any(dim='obs')
    
    n = len(region_idx_raw)
    print(f"Member {ENS_MEMB}: 2°S={reaches_2S.sum().values}/{n} ({reaches_2S.sum().values/n*100:.1f}%), 0°={reaches_0.sum().values}/{n} ({reaches_0.sum().values/n*100:.1f}%)")
    fracs_lens_raw.append(reaches_0.sum().values/n)

fracs_lens_raw = np.array(fracs_lens_raw)
print(f"\nLENS 0°: {fracs_lens_raw.mean()*100:.1f}% ± {fracs_lens_raw.std()*100:.1f}%")
print(f"FOSI 0°: 41.7%")

In [ ]:
# ============================================================
# FOSI - no truncation, 0° threshold
# ============================================================
sh_lats_f_raw = lat_f_raw.isel(trajectory=region_idx_f_raw)
sh_lons_f_raw = lon_f_raw.isel(trajectory=region_idx_f_raw)

wbc_count_f = 0
interior_count_f = 0

for i in range(len(region_idx_f_raw)):
    lat_traj = sh_lats_f_raw.isel(trajectory=i).values
    lon_traj = sh_lons_f_raw.isel(trajectory=i).values
    cross = np.where(lat_traj >= 0)[0]
    if len(cross) > 0:
        first = cross[0]
        if lon_traj[first] <= 165:
            wbc_count_f += 1
        else:
            interior_count_f += 1

frac_fosi_raw = (wbc_count_f + interior_count_f) / 60
print(f"FOSI total:    {wbc_count_f + interior_count_f}/60 = {frac_fosi_raw*100:.1f}%")
print(f"FOSI WBC:      {wbc_count_f}/60 = {wbc_count_f/60*100:.1f}%")
print(f"FOSI Interior: {interior_count_f}/60 = {interior_count_f/60*100:.1f}%")

# ============================================================
# LENS - no truncation, 0° threshold
# ============================================================
fracs_lens_raw = []
wbc_lens_raw = []
interior_lens_raw = []

for i, ENS_MEMB in enumerate(ENS_MEMBERS_LIST):
    file_path = '/glade/derecho/scratch/cassiacai/particle_trajectories/particle_trajectories_lens{}_SH_startingat{}m_1958_1977.zarr'.format(ENS_MEMB, INIT_DEPTH)
    lens_pt = xr.open_zarr(file_path)
    l_time, l_lon, l_lat, l_z = compute_data(lens_pt)
    
    region_idx_raw = get_region_lonlat(l_lat, l_lon, -40, -30, 215, 275)
    n_particles = len(region_idx_raw)
    
    sh_lats_l = l_lat.isel(trajectory=region_idx_raw)
    sh_lons_l = lon_f_raw.isel(trajectory=region_idx_raw)

    wbc_count = 0
    interior_count = 0
    for j in range(n_particles):
        lat_traj = sh_lats_l.isel(trajectory=j).values
        lon_traj = sh_lons_l.isel(trajectory=j).values
        cross = np.where(lat_traj >= 0)[0]
        if len(cross) > 0:
            first = cross[0]
            if lon_traj[first] <= 165:
                wbc_count += 1
            else:
                interior_count += 1

    frac = (wbc_count + interior_count) / n_particles
    fracs_lens_raw.append(frac)
    wbc_lens_raw.append(wbc_count / n_particles)
    interior_lens_raw.append(interior_count / n_particles)
    print(f"Member {ENS_MEMB:3d}: {wbc_count+interior_count}/{n_particles} ({frac*100:.1f}%) — WBC: {wbc_count} ({wbc_count/n_particles*100:.1f}%), Interior: {interior_count} ({interior_count/n_particles*100:.1f}%)")

fracs_lens_raw = np.array(fracs_lens_raw)
wbc_lens_raw = np.array(wbc_lens_raw)
interior_lens_raw = np.array(interior_lens_raw)

print(f"\n{'='*60}")
print(f"FOSI total:    {frac_fosi_raw*100:.1f}%")
print(f"LENS total:    {fracs_lens_raw.mean()*100:.1f}% ± {fracs_lens_raw.std()*100:.1f}%")
print(f"\nFOSI WBC:      {wbc_count_f/60*100:.1f}%")
print(f"LENS WBC:      {wbc_lens_raw.mean()*100:.1f}% ± {wbc_lens_raw.std()*100:.1f}%")
print(f"\nFOSI Interior: {interior_count_f/60*100:.1f}%")
print(f"LENS Interior: {interior_lens_raw.mean()*100:.1f}% ± {interior_lens_raw.std()*100:.1f}%")

In [ ]:
# FOSI - 0° with correct lon data
sh_lats_f_raw = lat_f_raw.isel(trajectory=region_idx_f_raw)
sh_lons_f_raw = lon_f_raw.isel(trajectory=region_idx_f_raw)

wbc_count_f = 0
interior_count_f = 0

for i in range(len(region_idx_f_raw)):
    lat_traj = sh_lats_f_raw.isel(trajectory=i).values
    lon_traj = sh_lons_f_raw.isel(trajectory=i).values
    cross = np.where(lat_traj >= 0)[0]
    if len(cross) > 0:
        first = cross[0]
        if lon_traj[first] <= 165:
            wbc_count_f += 1
        else:
            interior_count_f += 1

print(f"FOSI total:    {wbc_count_f + interior_count_f}/60 = {(wbc_count_f+interior_count_f)/60*100:.1f}%")
print(f"FOSI WBC:      {wbc_count_f}/60 = {wbc_count_f/60*100:.1f}%")
print(f"FOSI Interior: {interior_count_f}/60 = {interior_count_f/60*100:.1f}%")

# LENS - 0° with correct lon data
fracs_lens_raw = []
wbc_lens_raw = []
interior_lens_raw = []

for i, ENS_MEMB in enumerate(ENS_MEMBERS_LIST):
    file_path = '/glade/derecho/scratch/cassiacai/particle_trajectories/particle_trajectories_lens{}_SH_startingat{}m_1958_1977.zarr'.format(ENS_MEMB, INIT_DEPTH)
    lens_pt = xr.open_zarr(file_path)
    l_time, l_lon, l_lat, l_z = compute_data(lens_pt)
    
    region_idx_raw = get_region_lonlat(l_lat, l_lon, -40, -30, 215, 275)
    n_particles = len(region_idx_raw)
    
    sh_lats_l = l_lat.isel(trajectory=region_idx_raw)
    sh_lons_l = l_lon.isel(trajectory=region_idx_raw)  # fixed: use l_lon

    wbc_count = 0
    interior_count = 0
    for j in range(n_particles):
        lat_traj = sh_lats_l.isel(trajectory=j).values
        lon_traj = sh_lons_l.isel(trajectory=j).values
        cross = np.where(lat_traj >= 0)[0]
        if len(cross) > 0:
            first = cross[0]
            if lon_traj[first] <= 165:
                wbc_count += 1
            else:
                interior_count += 1

    frac = (wbc_count + interior_count) / n_particles
    fracs_lens_raw.append(frac)
    wbc_lens_raw.append(wbc_count / n_particles)
    interior_lens_raw.append(interior_count / n_particles)
    print(f"Member {ENS_MEMB:3d}: {wbc_count+interior_count}/{n_particles} ({frac*100:.1f}%) — WBC: {wbc_count} ({wbc_count/n_particles*100:.1f}%), Interior: {interior_count} ({interior_count/n_particles*100:.1f}%)")

fracs_lens_raw = np.array(fracs_lens_raw)
wbc_lens_raw = np.array(wbc_lens_raw)
interior_lens_raw = np.array(interior_lens_raw)

print(f"\n{'='*60}")
print(f"FOSI total:    {(wbc_count_f+interior_count_f)/60*100:.1f}%")
print(f"LENS total:    {fracs_lens_raw.mean()*100:.1f}% ± {fracs_lens_raw.std()*100:.1f}%")
print(f"\nFOSI WBC:      {wbc_count_f/60*100:.1f}%")
print(f"LENS WBC:      {wbc_lens_raw.mean()*100:.1f}% ± {wbc_lens_raw.std()*100:.1f}%")
print(f"\nFOSI Interior: {interior_count_f/60*100:.1f}%")
print(f"LENS Interior: {interior_lens_raw.mean()*100:.1f}% ± {interior_lens_raw.std()*100:.1f}%")

In [ ]:
fracs_lens = []
wbc_lens = []
interior_lens = []

for i in range(len(ENS_MEMBERS_LIST)):
    lat_l = lat_trunc_list[i]
    lon_l = lon_trunc_list[i]
    region_idx_l = region_indx_list[i]
    n_particles = len(region_idx_l)
    
    sh_lats_l = lat_l.isel(trajectory=region_idx_l)
    sh_lons_l = lon_l.isel(trajectory=region_idx_l)
    
    reaches = (sh_lats_l >= -2).any(dim='obs')  # <-- changed to 0
    frac = reaches.sum().values / n_particles
    fracs_lens.append(frac)
    
    wbc_count = 0
    interior_count = 0
    for j in range(n_particles):
        lat_traj = sh_lats_l.isel(trajectory=j).values
        lon_traj = sh_lons_l.isel(trajectory=j).values
        cross = np.where(lat_traj >= 0)[0]  # <-- changed to 0
        if len(cross) > 0:
            first = cross[0]
            if lon_traj[first] <= 165:
                wbc_count += 1
            else:
                interior_count += 1
    
    wbc_lens.append(wbc_count / n_particles)
    interior_lens.append(interior_count / n_particles)
    print(f"Member {ENS_MEMBERS_LIST[i]:3d}: {reaches.sum().values:2d}/{n_particles} ({frac*100:.1f}%) — WBC: {wbc_count} ({wbc_count/n_particles*100:.1f}%), Interior: {interior_count} ({interior_count/n_particles*100:.1f}%)")

fracs_lens = np.array(fracs_lens)
wbc_lens = np.array(wbc_lens)
interior_lens = np.array(interior_lens)

print(f"\nFOSI total:    28.3%")
print(f"LENS total:    {fracs_lens.mean()*100:.1f}% ± {fracs_lens.std()*100:.1f}%")
print(f"\nFOSI WBC:      26.7%")
print(f"LENS WBC:      {wbc_lens.mean()*100:.1f}% ± {wbc_lens.std()*100:.1f}%")
print(f"\nFOSI Interior: 1.7%")
print(f"LENS Interior: {interior_lens.mean()*100:.1f}% ± {interior_lens.std()*100:.1f}%")

In [ ]:
for i, ENS_MEMB in enumerate(ENS_MEMBERS_LIST):
    lat_l = lat_trunc_list[i]
    lon_l = lon_trunc_list[i]
    region_idx_l = region_indx_list[i]
    
    sh_lats_l = lat_l.isel(trajectory=region_idx_l)
    sh_lons_l = lon_l.isel(trajectory=region_idx_l)
    
    # Particles that never reach 2°S
    reaches = (sh_lats_l >= -2).any(dim='obs')
    stuck_mask = ~reaches
    
    # Final position of stuck particles
    stuck_lats = sh_lats_l.isel(trajectory=stuck_mask).isel(obs=-1)
    stuck_lons = sh_lons_l.isel(trajectory=stuck_mask).isel(obs=-1)
    
    print(f"Member {ENS_MEMB}: {stuck_mask.sum().values} stuck particles")
    print(f"  Final lat range: {stuck_lats.min().values:.1f} to {stuck_lats.max().values:.1f}")
    print(f"  Final lon range: {stuck_lons.min().values:.1f} to {stuck_lons.max().values:.1f}")
    print()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4),
                       subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.set_extent([120, 284, -50, 5], crs=ccrs.PlateCarree())
ax.axhline(y=-2, color='r', linestyle='--', linewidth=1, zorder=5, label='2°S threshold')
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)

colors = plt.cm.tab10(np.linspace(0, 1, len(ENS_MEMBERS_LIST)))

for i, ENS_MEMB in enumerate(ENS_MEMBERS_LIST):
    lat_l = lat_trunc_list[i]
    lon_l = lon_trunc_list[i]
    region_idx_l = region_indx_list[i]
    
    sh_lats_l = lat_l.isel(trajectory=region_idx_l)
    sh_lons_l = lon_l.isel(trajectory=region_idx_l)
    
    reaches = (sh_lats_l >= -5).any(dim='obs')
    stuck_mask = ~reaches
    
    stuck_lats = sh_lats_l.isel(trajectory=stuck_mask).isel(obs=-1).values
    stuck_lons = sh_lons_l.isel(trajectory=stuck_mask).isel(obs=-1).values
    
    ax.scatter(stuck_lons, stuck_lats, s=15, color=colors[i], 
               transform=ccrs.PlateCarree(), zorder=5, 
               label=f'Member {ENS_MEMB} (n={stuck_mask.sum().values})',
               alpha=0.7)

# Also plot FOSI stuck particles for comparison
sh_lats_f = lat_trunc_f.isel(trajectory=region_idx_f)
sh_lons_f = lon_trunc_f.isel(trajectory=region_idx_f)
reaches_f = (sh_lats_f >= -2).any(dim='obs')
stuck_mask_f = ~reaches_f
stuck_lats_f = sh_lats_f.isel(trajectory=stuck_mask_f).isel(obs=-1).values
stuck_lons_f = sh_lons_f.isel(trajectory=stuck_mask_f).isel(obs=-1).values
ax.scatter(stuck_lons_f, stuck_lats_f, s=30, color='k', marker='*',
           transform=ccrs.PlateCarree(), zorder=6, label=f'FOSI (n={stuck_mask_f.sum().values})')

gl = ax.gridlines(draw_labels={'left': True, 'bottom': True}, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 9}
gl.ylabel_style = {'size': 9}
ax.set_title('Final positions of particles not reaching 2°S', fontsize=11, fontweight='bold', loc='left')
plt.tight_layout()
plt.show()

In [ ]:
fracs_lens = []
wbc_lens = []
interior_lens = []

for i in range(len(ENS_MEMBERS_LIST)):
    lat_l = lat_trunc_list[i]
    lon_l = lon_trunc_list[i]
    region_idx_l = region_indx_list[i]
    n_particles = len(region_idx_l)
    
    sh_lats_l = lat_l.isel(trajectory=region_idx_l)
    sh_lons_l = lon_l.isel(trajectory=region_idx_l)
    
    # Fraction reaching 2°S
    reaches = (sh_lats_l >= -1).any(dim='obs')
    frac = reaches.sum().values / n_particles
    fracs_lens.append(frac)
    
    # Pathway breakdown
    wbc_count = 0
    interior_count = 0
    for j in range(n_particles):
        lat_traj = sh_lats_l.isel(trajectory=j).values
        lon_traj = sh_lons_l.isel(trajectory=j).values
        cross = np.where(lat_traj >= -2)[0]
        if len(cross) > 0:
            first = cross[0]
            if lon_traj[first] <= 165:
                wbc_count += 1
            else:
                interior_count += 1
    
    wbc_lens.append(wbc_count / n_particles)
    interior_lens.append(interior_count / n_particles)
    
    print(f"Member {ENS_MEMBERS_LIST[i]:3d}: {reaches.sum().values:2d}/{n_particles} reach 2°S ({frac*100:.1f}%) — WBC: {wbc_count} ({wbc_count/n_particles*100:.1f}%), Interior: {interior_count} ({interior_count/n_particles*100:.1f}%)")

fracs_lens = np.array(fracs_lens)
wbc_lens = np.array(wbc_lens)
interior_lens = np.array(interior_lens)

print(f"\n{'='*60}")
print(f"FOSI:  {frac_fosi*100:.1f}% reach 2°S")
print(f"LENS:  {fracs_lens.mean()*100:.1f}% ± {fracs_lens.std()*100:.1f}%")
print(f"\nFOSI  WBC pathway: {47/60*100:.1f}%")  # adjust 47 from your output
print(f"LENS  WBC pathway: {wbc_lens.mean()*100:.1f}% ± {wbc_lens.std()*100:.1f}%")
print(f"\nFOSI  Interior pathway: {6/60*100:.1f}%")  # adjust 6 from your output
print(f"LENS  Interior pathway: {interior_lens.mean()*100:.1f}% ± {interior_lens.std()*100:.1f}%")

In [ ]:
fracs_lens = []

for i in range(len(ENS_MEMBERS_LIST)):
    lat_l = lat_trunc_list[i]           # shape (400, 244)
    region_idx_l = region_indx_list[i]  # SH particle indices
    n_particles = len(region_idx_l)
    
    sh_lats_l = lat_l.isel(trajectory=region_idx_l)          # (n_particles, 244)
    reaches = (sh_lats_l >= -2).any(dim='obs')                # True/False
    frac = reaches.sum().values / n_particles
    fracs_lens.append(frac)
    print(f"Member {ENS_MEMBERS_LIST[i]}: {reaches.sum().values}/{n_particles} ({frac*100:.1f}%)")

fracs_lens = np.array(fracs_lens)
print(f"\nLENS mean: {fracs_lens.mean()*100:.1f}%")
print(f"LENS std:  {fracs_lens.std()*100:.1f}%")
print(f"\nFOSI:      {frac_fosi*100:.1f}%")
print(f"LENS:      {fracs_lens.mean()*100:.1f}% ± {fracs_lens.std()*100:.1f}%")

In [ ]:
da = da_f
lon_data= lon_trunc_f[:, 0:244]
lat_data= lat_trunc_f[:, 0:244]
z_data= z_trunc_f[:, 0:244]
region_idx = region_idx_f

fig, ax = plt.subplots(figsize=(4, 3),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -50, 5], crs=ccrs.PlateCarree())
# ax.set_extent([120, 284, -5, 50], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

####### INITIAL LOCATIONS
ax.scatter(lon_data[region_idx, 0], lat_data[region_idx, 0], 
           s=0.5, c='k', marker='o', edgecolor='k', zorder=100,transform=ccrs.PlateCarree())

# # #### CONTOURF AND CONTOUR
# contourf_plot = da.plot.contourf(
#     x='lon', y='lat', cmap='Blues', add_colorbar=False, transform=ccrs.PlateCarree(),
#     levels=11, vmax=60e-5, vmin=0, alpha=1) # 60 for SH

#### TRAJECTORIES
for ind in range(60):
    print(ind)
    i = region_idx[::1][ind]
    lon = lon_data.isel(trajectory=i).values
    lat = lat_data.isel(trajectory=i).values
    depth = FOSI_pd_obs_array[ind,:]#z_data.isel(trajectory=i).values
    points = np.array([lon, lat]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    lc = mcollections.LineCollection(
            segments, cmap='jet', norm=plt.Normalize(1.025, 1.026), 
            linewidth=.8, alpha=0.8, transform=ccrs.PlateCarree())
    lc.set_array(depth[:-1])
    ax.add_collection(lc)

#### TITLE
# plt.title('(f) FOSI \n30°S to 45°S, 215°E to 275°E', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.title('(a) FOSI: South Pacific - PD', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

In [ ]:
da = da_f
lon_data= lon_trunc_f[:, 0:244]
lat_data= lat_trunc_f[:, 0:244]
z_data= z_trunc_f[:, 0:244]
region_idx = region_idx_f

fig, ax = plt.subplots(figsize=(4, 3),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -50, 5], crs=ccrs.PlateCarree())
# ax.set_extent([120, 284, -5, 50], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

####### INITIAL LOCATIONS
ax.scatter(lon_data[region_idx, 0], lat_data[region_idx, 0], 
           s=0.5, c='k', marker='o', edgecolor='k', zorder=100,transform=ccrs.PlateCarree())

# # #### CONTOURF AND CONTOUR
# contourf_plot = da.plot.contourf(
#     x='lon', y='lat', cmap='Blues', add_colorbar=False, transform=ccrs.PlateCarree(),
#     levels=11, vmax=60e-5, vmin=0, alpha=1) # 60 for SH

#### TRAJECTORIES
for ind in range(60):
    i = region_idx[::1][ind]
    lon = lon_data.isel(trajectory=i).values
    lat = lat_data.isel(trajectory=i).values
    depth = ceil_array[ind,:]#z_data.isel(trajectory=i).values
    points = np.array([lon, lat]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    lc = mcollections.LineCollection(
            segments, cmap='brg', norm=plt.Normalize(0, 20), 
            linewidth=.8, alpha=0.8, transform=ccrs.PlateCarree())
    lc.set_array(depth[:-1])
    ax.add_collection(lc)

#### TITLE
# plt.title('(f) FOSI \n30°S to 45°S, 215°E to 275°E', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.title('(d) FOSI: South Pacific - Timing', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

In [ ]:
da = da_f
lon_data= lon_trunc_f[:, 0:244]
lat_data= lat_trunc_f[:, 0:244]
z_data= z_trunc_f[:, 0:244]
region_idx = region_idx_f

fig, ax = plt.subplots(figsize=(4, 3),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
# ax.set_extent([120, 284, -50, 5], crs=ccrs.PlateCarree())
ax.set_extent([120, 160, -50, 5], crs=ccrs.PlateCarree())
# ax.set_extent([120, 284, -5, 50], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

####### INITIAL LOCATIONS
ax.scatter(lon_data[region_idx, 0], lat_data[region_idx, 0], 
           s=0.5, c='k', marker='o', edgecolor='k', zorder=100,transform=ccrs.PlateCarree())

# # #### CONTOURF AND CONTOUR
# contourf_plot = da.plot.contourf(
#     x='lon', y='lat', cmap='Blues', add_colorbar=False, transform=ccrs.PlateCarree(),
#     levels=11, vmax=60e-5, vmin=0, alpha=1) # 60 for SH

#### TRAJECTORIES
for i in region_idx[::1]:
    lon = lon_data.isel(trajectory=i).values
    lat = lat_data.isel(trajectory=i).values
    depth = z_data.isel(trajectory=i).values
    points = np.array([lon, lat]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    lc = mcollections.LineCollection(
            segments, cmap='jet', norm=plt.Normalize(0, 25000), 
            linewidth=.8, alpha=1, transform=ccrs.PlateCarree())
    lc.set_array(depth[:-1])
    ax.add_collection(lc)

#### TITLE
# plt.title('(f) FOSI \n30°S to 45°S, 215°E to 275°E', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.title('(b) FOSI: South Pacific', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

In [ ]:
# LENS --- South Pacific
ENS_MEMBERS_LIST = [0, 65,  32, 85, 61, 90, 80, 68, 73, 49] 
INIT_DEPTH = 50

region_indx_list = []; da_list = []
lon_trunc_list = []; lat_trunc_list = []; z_trunc_list = []

for ENS_MEMB in ENS_MEMBERS_LIST:
    file_path = '/glade/derecho/scratch/cassiacai/particle_trajectories/particle_trajectories_lens{}_SH_startingat{}m_1958_1977.zarr'.format(
        ENS_MEMB, INIT_DEPTH)

    lens_pt = xr.open_zarr(file_path)
    l_time, l_lon, l_lat, l_z = compute_data(lens_pt)
    print(ENS_MEMB)
    
    # ### SOUTH PACIFIC
    # da_l, region_idx_l,lon_trunc_l,lat_trunc_l, z_trunc_l = calc_density(
    #    lon_data=l_lon, lat_data=l_lat, z_data=l_z, 
    #     lat_min=-40, lat_max=-30, lon_min=220, lon_max=240,
    #     timestart=0, timeend=244)

    ### SOUTH PACIFIC
    da_l, region_idx_l,lon_trunc_l,lat_trunc_l, z_trunc_l = calc_density(
       lon_data=l_lon, lat_data=l_lat, z_data=l_z, 
        lat_min=-40, lat_max=-30, lon_min=215, lon_max=275, # 220 to 240
    # timestart=0, timeend=244)
    timestart=244-61, timeend=244)

    
    region_indx_list.append(region_idx_l)
    lon_trunc_list.append(lon_trunc_l)
    lat_trunc_list.append(lat_trunc_l)
    z_trunc_list.append(z_trunc_l)
    da_list.append(da_l)

In [ ]:
LENS61_pd_obs_array = np.load('/glade/derecho/scratch/cassiacai/LENS_SH61_traj_PD_fullarray.npy')


In [ ]:
LENS61_pd_obs_array.shape

In [ ]:
original_array = np.linspace(0, 20, 244)

array_2d = np.tile(original_array, (60, 1))
ceil_array = np.ceil(array_2d).astype(int)


In [ ]:
ceil_array

In [ ]:
da = da_xr.mean(dim='ensemble')
lon_data= lon_trunc_list[4][ :, 0:244]
lat_data= lat_trunc_list[4][:, 0:244]
z_data= z_trunc_list[4][:, 0:244]
region_idx = region_indx_list[0]

fig, ax = plt.subplots(figsize=(4, 3),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -50, 5], crs=ccrs.PlateCarree())
# ax.set_extent([120, 284, -5, 50], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

####### INITIAL LOCATIONS
ax.scatter(lon_data[region_idx, 0], lat_data[region_idx, 0], 
           s=0.5, c='k', marker='o', edgecolor='k', zorder=100,transform=ccrs.PlateCarree())

# # #### CONTOURF AND CONTOUR
# contourf_plot = da.plot.contourf(
#     x='lon', y='lat', cmap='Blues', add_colorbar=False, transform=ccrs.PlateCarree(),
#     levels=11, vmax=60e-5, vmin=0, alpha=1) # 60 for SH

#### TRAJECTORIES
for ind in range(60):
    i = region_idx[::1][ind]
    lon = lon_data.isel(trajectory=i).values
    lat = lat_data.isel(trajectory=i).values
    depth = ceil_array[ind,:]#z_data.isel(trajectory=i).values
    points = np.array([lon, lat]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    lc = mcollections.LineCollection(
            segments, cmap='brg', norm=plt.Normalize(0, 20), 
            linewidth=.8, alpha=0.8, transform=ccrs.PlateCarree())
    lc.set_array(depth[:-1])
    ax.add_collection(lc)

#### TITLE
# plt.title('(f) FOSI \n30°S to 45°S, 215°E to 275°E', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.title('(f) LENS: South Pacific - Timing', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

In [ ]:
da = da_xr.mean(dim='ensemble')
lon_data= lon_trunc_list[4][ :, 0:244]
lat_data= lat_trunc_list[4][:, 0:244]
z_data= z_trunc_list[4][:, 0:244]
region_idx = region_indx_list[0]

fig, ax = plt.subplots(figsize=(4, 3),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -50, 5], crs=ccrs.PlateCarree())
# ax.set_extent([120, 284, -5, 50], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

####### INITIAL LOCATIONS
ax.scatter(lon_data[region_idx, 0], lat_data[region_idx, 0], 
           s=0.5, c='k', marker='o', edgecolor='k', zorder=100,transform=ccrs.PlateCarree())

# # #### CONTOURF AND CONTOUR
# contourf_plot = da.plot.contourf(
#     x='lon', y='lat', cmap='Blues', add_colorbar=False, transform=ccrs.PlateCarree(),
#     levels=11, vmax=60e-5, vmin=0, alpha=1) # 60 for SH

#### TRAJECTORIES
for ind in range(60):
    print(ind)
    i = region_idx[::1][ind]
    lon = lon_data.isel(trajectory=i).values
    lat = lat_data.isel(trajectory=i).values
    depth = LENS61_pd_obs_array[ind,:]#z_data.isel(trajectory=i).values
    points = np.array([lon, lat]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    lc = mcollections.LineCollection(
            segments, cmap='jet', norm=plt.Normalize(1.025, 1.026), 
            linewidth=.8, alpha=0.8, transform=ccrs.PlateCarree())
    lc.set_array(depth[:-1])
    ax.add_collection(lc)

#### TITLE
# plt.title('(f) FOSI \n30°S to 45°S, 215°E to 275°E', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.title('(b) LENS: South Pacific - PD', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(0.2, 3))

# Blues colorbar
density_sm1 = plt.cm.ScalarMappable(cmap='brg', norm=plt.Normalize(0, 20))
cbar1 = plt.colorbar(density_sm1, cax=ax1, orientation='vertical')
cbar1.set_label('Time (years)', fontsize=10)
tick_values1 = np.linspace(0, 20, 11)  # 0, 0.5, 1.0, 1.5, 2.0
cbar1.set_ticks(tick_values1)
cbar1.set_ticklabels([f'{x:.0f}' for x in tick_values1])
cbar1.ax.tick_params(labelsize=10)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(0.2, 3))

# Blues colorbar
density_sm1 = plt.cm.ScalarMappable(cmap='jet', norm=plt.Normalize(1.025, 1.026))
cbar1 = plt.colorbar(density_sm1, cax=ax1, orientation='vertical')
cbar1.set_label('Potential Density (kg/m3)', fontsize=10)
tick_values1 = np.linspace(1.025, 1.026, 11)  # 0, 0.5, 1.0, 1.5, 2.0
cbar1.set_ticks(tick_values1)
cbar1.set_ticklabels([f'{x:.04f}' for x in tick_values1])
cbar1.ax.tick_params(labelsize=10)

plt.tight_layout()
plt.show()

In [ ]:
np.linspace(1.025, 1.026, 11)

In [ ]:
# # LENS --- North Pacific
# ENS_MEMBERS_LIST = [0, 65,  32, 85, 61, 90, 80, 68, 73, 49] 
# INIT_DEPTH = 5

# region_indx_list = []; da_list = []
# lon_trunc_list = []; lat_trunc_list = []; z_trunc_list = []

# for ENS_MEMB in ENS_MEMBERS_LIST:
#     print(ENS_MEMB)
#     file_path = '/glade/derecho/scratch/cassiacai/particle_trajectories/particle_trajectories_lens{}_NH_startingat{}m_1958_1977.zarr'.format(
#         ENS_MEMB, INIT_DEPTH)

#     lens_pt = xr.open_zarr(file_path)
#     l_time, l_lon, l_lat, l_z = compute_data(lens_pt)

#     ### NORTH PACIFIC
#     da_l, region_idx_l,lon_trunc_l,lat_trunc_l, z_trunc_l = calc_density(
#        lon_data=l_lon, lat_data=l_lat, z_data=l_z, 
#         lat_min=38, lat_max=45, lon_min=140, lon_max=210,
#         timestart=244-61, timeend=244)
    
#     region_indx_list.append(region_idx_l)
#     lon_trunc_list.append(lon_trunc_l)
#     lat_trunc_list.append(lat_trunc_l)
#     z_trunc_list.append(z_trunc_l)
#     da_list.append(da_l)

#     # #### Leakage into the Indian Ocean
#     # lon_data = l_lon
#     # lat_data = l_lat
#     # region_idx = region_idx_l
    
#     # no_in_indotf = len(xr.where(lon_data[region_idx, :].min(axis=1) < 100, 1., np.nan).dropna(dim='trajectory'))
#     # perc = (no_in_indotf / len(region_idx))*100
#     # print(perc)

In [ ]:
# # LENS - leakage into the Indian Ocean (West of 110°E)
# leakage_50m = [11.666666666666666, 15.0, 5.0, 5.0, 11.666666666666666, 13.333333333333334, 5.0, 18.333333333333332, 15., 11.666666666666666]
# print(np.mean(leakage_50m))
# print(np.std(leakage_50m))

# leakage_5m = [5., 6.7, 1.7, 1.7, 5., 6.7, 15., 3.3, 8.3]
# print(np.mean(leakage_5m))
# print(np.std(leakage_5m))

In [ ]:
da_xr = xr.concat(da_list, dim='ensemble')

In [ ]:
da = da_xr.mean(dim='ensemble')
lon_data= lon_trunc_list[4][ :, 0:244]
lat_data= lat_trunc_list[4][:, 0:244]
z_data= z_trunc_list[4][:, 0:244]
region_idx = region_indx_list[0]

fig, ax = plt.subplots(figsize=(4, 3),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -50, 5], crs=ccrs.PlateCarree())
# ax.set_extent([120, 284, -5, 50], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

ax.scatter(lon_data[region_idx, 0], lat_data[region_idx, 0], 
           s=0.5, c='k', marker='o', edgecolor='k', zorder=100,transform=ccrs.PlateCarree())

# # #### CONTOURF AND CONTOUR
# contourf_plot = da.plot.contourf(
#     x='lon', y='lat', cmap='Blues', add_colorbar=False, transform=ccrs.PlateCarree(),
#     levels=11, vmax=60e-5, vmin=0, alpha=1) # 60 for SH

#### TRAJECTORIES
for i in region_indx_list[5][::1]:
    lon = lon_trunc_list[5].isel(trajectory=i).values
    lat = lat_trunc_list[5].isel(trajectory=i).values
    depth = z_trunc_list[5].isel(trajectory=i).values
    points = np.array([lon, lat]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    lc = mcollections.LineCollection(
            segments, cmap='jet', norm=plt.Normalize(0, 25000), 
            linewidth=0.8, alpha=1, transform=ccrs.PlateCarree())
    lc.set_array(depth[:-1])
    ax.add_collection(lc)

# #### COLORBAR
# density_sm = plt.cm.ScalarMappable(cmap='Blues', norm=plt.Normalize(0.5e-5, 80.5e-5))
# density_sm.set_array([])
# density_cbar = plt.colorbar(density_sm, ax=ax, aspect=30, shrink=0.8, pad=0.1, orientation='horizontal')
# density_cbar.set_label('Normalized Trajectory Density', fontsize=11)
# tick_values = np.arange(0, 90, 20)  # This gives 5, 10, 15, 20, 25, 30
# density_cbar.set_ticks(tick_values * 1e-5)
# density_cbar.set_ticklabels([f'{x}' for x in tick_values])  # Shows as 5, 10, 15, etc.
# density_cbar.ax.tick_params(labelsize=12)

#### TITLE
# plt.title('LENS: South Pacific \n25°S to 45°S, 250°E to 275°E', fontsize=11, fontweight='bold', zorder=12, loc='left')

plt.title('(d) LENS: South Pacific', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

In [ ]:
da_diff_xr = da_xr - da_f
da_diff_xr.mean(dim=('y','x')).argmin()

In [ ]:
da_diff = da_l - da_f

fig, ax = plt.subplots(figsize=(4, 3), #4, 3
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -50, 5], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)

ax.scatter(lon_trunc_f[region_idx, 0][::1], lat_trunc_f[region_idx, 0][::1], 
           s=2, c='k', marker='o', edgecolor='k', zorder=200,transform=ccrs.PlateCarree())

# # #### CONTOURF AND CONTOUR
# contourf_plot = da_diff.plot.contourf(
#     x='lon', y='lat', cmap='PRGn', add_colorbar=False, transform=ccrs.PlateCarree(),
#     alpha=1, vmin=-0.0002, vmax=0.0002, levels=12)

# #### CONTOURF AND CONTOUR
contourf_plot = da_diff.plot.contourf(
    x='lon', y='lat', cmap='PRGn', add_colorbar=False, transform=ccrs.PlateCarree(),
    alpha=1, vmin=-30e-5, vmax=30e-5, levels=13)

# CONTOUR LINES
da_diff.plot.contour(
    x='lon', y='lat', transform=ccrs.PlateCarree(), colors='k', linewidths=1,
    levels=[-20e-5, -15e-5, -10e-5, -5e-5, 5e-5, 10e-5, 15e-5, 20e-5], alpha=1)

plt.title('(f) LENS - FOSI: Y0-20', fontsize=12, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

In [ ]:
diff_PV_bin_prog_std = diff_PV.sel(lat=slice(-40, 40), lon=slice(120,290))

In [ ]:
diff_PV_bin_prog = diff_PV.mean(dim='ensemble').sel(lat=slice(-40, 40), lon=slice(120,290))
diff_SH_PV_binarize = xr.where(diff_PV_bin_prog > diff_PV_bin_prog_std.std()/10, 1., 0.)
diff_NH_PV_binarize = xr.where(diff_PV_bin_prog > 0.15e-11, 1., 0.)

In [ ]:
diff_SH_PV_binarize.plot.contourf()

In [ ]:
print(diff_PV_bin_prog_std.std())
0.03e-11

In [ ]:
fosi_nan = da_f.fillna(0)
lens_nan = da_xr.mean(dim='ensemble').fillna(0)

diff = lens_nan - fosi_nan

fig, ax = plt.subplots(figsize=(4, 3), 
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -50, 5], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')  # Override the equal aspect ratio

####### DENSITY CONTOUR
contourf_plot = diff.plot.contourf(
    x='lon', y='lat', cmap='PRGn', add_colorbar=False, transform=ccrs.PlateCarree(),
    levels=40, 
    vmax=30e-5, 
    vmin=-30e-5, alpha=1
)

# regridded_LENS_PV.isel(ensemble=ENS_IND).sel(lat=slice(-25, 25), lon=slice(120,290)).plot.contour(cmap='cyan',
#     levels=[0.4e-11, 0.45e-11, 0.5e-11],transform=ccrs.PlateCarree(), add_colorbar=False, zorder=400)

# CONTOUR LINES
diff.plot.contour(
    x='lon', y='lat', transform=ccrs.PlateCarree(), colors='k', linewidths=1,
    levels=[-20e-5, -15e-5, -10e-5, -5e-5, 5e-5, 10e-5, 15e-5, 20e-5], alpha=1)

plt.title('(h)', fontsize=12, fontweight='bold', zorder=12, loc='left')

# diff_PV_binarize.sel(lat=slice(-15, 0), lon=slice(120,240)).plot.contourf(
#     cmap='Grays',transform=ccrs.PlateCarree(), add_colorbar=False, alpha=0.2)
# cs = diff_PV_binarize.sel(lat=slice(-15, 5), lon=slice(120,290)).plot.contourf(
#     transform=ccrs.PlateCarree(), add_colorbar=False, alpha=0.1,
#     hatches=['','xxxx', ''],  # Slanted lines pattern
#     colors='white', levels=[0, 0.5, 1.5]
# )

cs = diff_SH_PV_binarize.sel(lat=slice(-20, 5), lon=slice(120,290)).plot.contourf(
    transform=ccrs.PlateCarree(), add_colorbar=False, alpha=0.15,
    hatches=['','xxxx', ''],
    colors='white', levels=[0, 0.5, 1.5]
)

ax.scatter(lon_data[region_idx, 0], lat_data[region_idx, 0], 
           s=0.5, c='k', marker='o', edgecolor='k', zorder=100,transform=ccrs.PlateCarree())

plt.title('')
plt.title('(h) LENS - FOSI: Y15-20', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.tight_layout()
plt.show()

In [ ]:
# LENS --- EQ THERMOCLINE
ENS_MEMBERS_LIST = [0, 65,  32, 85, 61, 90, 80, 68, 73, 49] #[0, 65,  32, 85, 61, 90, 80, 68, 73, 49] 

region_indx_list = []; da_list = []
lon_trunc_list = []; lat_trunc_list = []; z_trunc_list = []

for ENS_MEMB in ENS_MEMBERS_LIST:
    print(ENS_MEMB)
    file_path = "/glade/derecho/scratch/cassiacai/particle_trajectories/particle_trajectories_lens{}_EQ_startingat5m_1958_1977.zarr".format(
        ENS_MEMB)

    lens_pt = xr.open_zarr(file_path)
    l_time, l_lon, l_lat, l_z = compute_data(lens_pt)

    ### NORTH PACIFIC
    da_l, region_idx_l,lon_trunc_l,lat_trunc_l, z_trunc_l = calc_density_eqthermocline(
       lon_data=l_lon, lat_data=l_lat, z_data=l_z, 
        lat_min=-1, lat_max=1, lon_min=140, lon_max=280,
        timestart=0, timeend=244)
    
    region_indx_list.append(region_idx_l)
    lon_trunc_list.append(lon_trunc_l)
    lat_trunc_list.append(lat_trunc_l)
    z_trunc_list.append(z_trunc_l)
    da_list.append(da_l)

In [ ]:
da_xr = xr.concat(da_list, dim='ensemble')

In [ ]:
da = da_xr.mean(dim='ensemble')
lon_data= lon_trunc_list[0][ :, 0:244]
lat_data= lat_trunc_list[0][:, 0:244]
z_data= z_trunc_list[0][:, 0:244]
region_idx = region_indx_list[0]

fig, ax = plt.subplots(figsize=(4, 4),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -40, 40], crs=ccrs.PlateCarree())
# ax.set_extent([120, 284, -5, 50], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

ax.scatter(lon_trunc_f[region_idx, 0][::2], lat_trunc_f[region_idx, 0][::2], 
           s=5, c='k', marker='o', edgecolor='k', zorder=200,transform=ccrs.PlateCarree())

# #### CONTOURF AND CONTOUR
contourf_plot = da.plot.contourf(
    x='lon', y='lat', cmap='Blues', add_colorbar=False, transform=ccrs.PlateCarree(),
    levels=11, vmax=40e-5, vmin=0, alpha=1) # 60 for SH

#### TRAJECTORIES
for i in region_indx_list[0][::4]:
    lon = lon_trunc_list[0].isel(trajectory=i).values
    lat = lat_trunc_list[0].isel(trajectory=i).values
    depth = z_trunc_list[0].isel(trajectory=i).values
    points = np.array([lon, lat]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    lc = mcollections.LineCollection(
            segments, cmap='jet', norm=plt.Normalize(0, 25000), 
            linewidth=1., alpha=1, transform=ccrs.PlateCarree())
    lc.set_array(depth[:-1])
    ax.add_collection(lc)

#### COLORBAR
density_sm = plt.cm.ScalarMappable(cmap='Blues', norm=plt.Normalize(0.5e-5, 80.5e-5))
density_sm.set_array([])
density_cbar = plt.colorbar(density_sm, ax=ax, aspect=30, shrink=0.8, pad=0.1, orientation='horizontal')
density_cbar.set_label('Normalized Trajectory Density', fontsize=11)
tick_values = np.arange(0, 90, 20)  # This gives 5, 10, 15, 20, 25, 30
density_cbar.set_ticks(tick_values * 1e-5)
density_cbar.set_ticklabels([f'{x}' for x in tick_values])  # Shows as 5, 10, 15, etc.
density_cbar.ax.tick_params(labelsize=12)

#### TITLE
plt.title('(g) LENS', fontsize=12, fontweight='bold', zorder=12, loc='left')
plt.show()

In [ ]:
filepath = "/glade/derecho/scratch/cassiacai/particle_trajectories/particle_trajectories_fosi{}_EQ_startingat5m_1958_1977.zarr"
fosi_data_xarray = xr.open_zarr(filepath)
f_time, f_lon, f_lat, f_z = compute_data(fosi_data_xarray)

In [ ]:
da_f, region_idx_f, lon_trunc_f,lat_trunc_f, z_trunc_f = calc_density_eqthermocline(
   lon_data=f_lon, lat_data=f_lat, z_data=f_z, 
    lat_min=-1, lat_max=1, 
    lon_min=140, lon_max=280, # 200, 260
    timestart=0, timeend=244)

In [ ]:
da = da_f
lon_data= lon_trunc_f[:, 0:244]
lat_data= lat_trunc_f[:, 0:244]
z_data= z_trunc_f[:, 0:244]
region_idx = region_idx_f

fig, ax = plt.subplots(figsize=(4, 4), #4, 3
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -40, 40], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

ax.scatter(lon_trunc_f[region_idx, 0][::2], lat_trunc_f[region_idx, 0][::2], 
           s=5, c='k', marker='o', edgecolor='k', zorder=200,transform=ccrs.PlateCarree())

# #### CONTOURF AND CONTOUR
contourf_plot = da_f.plot.contourf(
    x='lon', y='lat', cmap='Blues', add_colorbar=False, transform=ccrs.PlateCarree(),
    levels=11, vmax=40e-5, vmin=0, alpha=1)

#### TRAJECTORIES
for i in region_idx[::4]:
    lon = lon_data.isel(trajectory=i).values
    lat = lat_data.isel(trajectory=i).values
    depth = z_data.isel(trajectory=i).values
    points = np.array([lon, lat]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    lc = mcollections.LineCollection(
            segments, cmap='jet', norm=plt.Normalize(0, 25000), 
            linewidth=1, alpha=0.8, transform=ccrs.PlateCarree())
    lc.set_array(depth[:-1])
    ax.add_collection(lc)

#### COLORBAR
density_sm = plt.cm.ScalarMappable(cmap='Blues', norm=plt.Normalize(0e-5, 40e-5))
density_sm.set_array([])
density_cbar = plt.colorbar(density_sm, ax=ax, aspect=30, shrink=0.8, pad=0.1, orientation='horizontal')
density_cbar.set_label('Normalized Trajectory Density', fontsize=11)
tick_values = np.arange(0, 45, 5)  # This gives 5, 10, 15, 20, 25, 30
density_cbar.set_ticks(tick_values * 1e-5)
density_cbar.set_ticklabels([f'{x}' for x in tick_values])  # Shows as 5, 10, 15, etc.
density_cbar.ax.tick_params(labelsize=12)

plt.title('(h) FOSI', fontsize=12, fontweight='bold', zorder=12, loc='left')
plt.show()

In [ ]:
da_diff = da_l - da_f

fig, ax = plt.subplots(figsize=(4, 4), #4, 3
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -40, 40], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)

ax.scatter(lon_trunc_f[region_idx, 0][::2], lat_trunc_f[region_idx, 0][::2], 
           s=5, c='k', marker='o', edgecolor='k', zorder=200,transform=ccrs.PlateCarree())

# # #### CONTOURF AND CONTOUR
# contourf_plot = da_diff.plot.contourf(
#     x='lon', y='lat', cmap='PRGn', add_colorbar=False, transform=ccrs.PlateCarree(),
#     alpha=1, vmin=-0.0002, vmax=0.0002, levels=12)

# #### CONTOURF AND CONTOUR
contourf_plot = da_diff.plot.contourf(
    x='lon', y='lat', cmap='PRGn', add_colorbar=False, transform=ccrs.PlateCarree(),
    alpha=1, vmin=-30e-5, vmax=30e-5, levels=21)

da_diff.plot.contour(
    x='lon', y='lat', transform=ccrs.PlateCarree(), colors='k', linewidths=1,
    levels=[-20e-5, -15e-5, -10e-5, -5e-5, 5e-5, 10e-5, 15e-5, 20e-5], alpha=1)

#### COLORBAR
density_sm = plt.cm.ScalarMappable(cmap='PRGn', norm=plt.Normalize(-30e-5, 30e-5))
density_sm.set_array([])
density_cbar = plt.colorbar(density_sm, ax=ax, aspect=30, shrink=0.8, pad=0.1, orientation='horizontal')
density_cbar.set_label('Normalized Trajectory Density', fontsize=11)
tick_values = np.arange(-30, 35, 10)  # This gives 5, 10, 15, 20, 25, 30
density_cbar.set_ticks(tick_values * 1e-5)
density_cbar.set_ticklabels([f'{x}' for x in tick_values])  # Shows as 5, 10, 15, etc.
density_cbar.ax.tick_params(labelsize=12)

plt.title('(i) LENS - FOSI', fontsize=12, fontweight='bold', zorder=12, loc='left')
plt.show()

In [ ]:
da = da_l
lon_data= lon_trunc_l[:, 0:244]
lat_data= lat_trunc_l[:, 0:244]
z_data= z_trunc_l[:, 0:244]
region_idx = region_idx_l

fig, ax = plt.subplots(figsize=(4, 3), #4, 3
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -40, 40], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
# ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

ax.scatter(lon_trunc_f[region_idx, 0][::2], lat_trunc_f[region_idx, 0][::2], 
           s=5, c='k', marker='o', edgecolor='k', zorder=200,transform=ccrs.PlateCarree())

# #### CONTOURF AND CONTOUR
contourf_plot = da_f.plot.contourf(
    x='lon', y='lat', cmap='Blues', add_colorbar=False, transform=ccrs.PlateCarree(),
    levels=16, vmax=30e-5, vmin=0, alpha=1)

#### TRAJECTORIES
for i in region_idx[::2]:
    lon = lon_data.isel(trajectory=i).values
    lat = lat_data.isel(trajectory=i).values
    depth = z_data.isel(trajectory=i).values
    points = np.array([lon, lat]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    lc = mcollections.LineCollection(
            segments, cmap='jet', norm=plt.Normalize(0, 25000), 
            linewidth=0.5, alpha=0.8, transform=ccrs.PlateCarree())
    lc.set_array(depth[:-1])
    ax.add_collection(lc)

#### COLORBAR
density_sm = plt.cm.ScalarMappable(cmap='Blues', norm=plt.Normalize(0e-5, 30e-5))
density_sm.set_array([])
density_cbar = plt.colorbar(density_sm, ax=ax, aspect=30, shrink=0.8, pad=0.1, orientation='horizontal')
density_cbar.set_label('Normalized Trajectory Density', fontsize=11)
tick_values = np.arange(0, 35, 5)  # This gives 5, 10, 15, 20, 25, 30
density_cbar.set_ticks(tick_values * 1e-5)
density_cbar.set_ticklabels([f'{x}' for x in tick_values])  # Shows as 5, 10, 15, etc.
density_cbar.ax.tick_params(labelsize=12)

# plt.title('(b)', fontsize=15, fontweight='bold', zorder=12, loc='left')
plt.show()

## Processing

In [ ]:
def regrid_time_mean_popvar(ds, regridder, time_start, time_end):
    var_mean = ds.isel(time=slice(time_start,time_end)).isel(TEMP = 2).mean(dim='time').compute()
    var_1deg = proc_utils.regrid_SMYLE(var_mean)
    regridded_var = regridder(var_1deg)
    return regridded_var

In [ ]:
time_start = 0; time_end = 240
# time_start = 456; time_end = 696

In [ ]:
ensemble_members = [0, 65,  32, 85, 61, 90, 80, 68, 73, 49]

In [ ]:
CESMLENS_var = afuncs.LENS_for_regridding()

## LENS data

In [ ]:
##### ------------  LENS
##### POTENTIAL VORTICITY
file_paths = [f'/glade/derecho/scratch/cassiacai/regridded_PV_{member}.nc' for member in ensemble_members]
LENS_PV_all = xr.open_mfdataset(file_paths, combine='nested', concat_dim='ensemble')
LENS_PV_mean = LENS_PV_all.PV.isel(time=slice(time_start,time_end)).isel(TEMP = 2).mean(dim='time').compute()
LENS_PV_1deg = proc_utils.regrid_SMYLE(LENS_PV_mean)

regridder = xe.Regridder(LENS_PV_1deg, CESMLENS_var[:,:], 'nearest_s2d', periodic=True)

regridded_LENS_PV = np.absolute(regridder(LENS_PV_1deg))

##### POTENTIAL DENSITY
file_paths = [f'/glade/derecho/scratch/cassiacai/regridded_PD_{member}.nc' for member in ensemble_members]
LENS_PD_all = xr.open_mfdataset(file_paths, combine='nested', concat_dim='ensemble')

ds = LENS_PD_all.PD
regridded_LENS_PD = regrid_time_mean_popvar(ds, regridder, time_start, time_end)
regridded_LENS_PD_nan = xr.where(regridded_LENS_PD == 0., np.nan, regridded_LENS_PD)

##### TEMPERATURE
file_paths = [f'/glade/derecho/scratch/cassiacai/regridded_TEMP_{member}.nc' for member in ensemble_members]
LENS_TEMP_all = xr.open_mfdataset(file_paths, combine='nested', concat_dim='ensemble')

ds = LENS_TEMP_all.__xarray_dataarray_variable__
regridded_LENS_TEMP = regrid_time_mean_popvar(ds, regridder, time_start, time_end)
regridded_LENS_TEMP_nan = xr.where(regridded_LENS_TEMP == 0., np.nan, regridded_LENS_TEMP)

# ##### SST
# SST = atm_var_ens(ENS_MEMB, 'SST')
# SST_nan = xr.where(SST == 0., np.nan, SST) - 273.15

In [ ]:
##### ------------  LENS
##### POTENTIAL VORTICITY
file_paths = [f'/glade/derecho/scratch/cassiacai/regridded_PV_{member}.nc' for member in ensemble_members]
LENS_PV_all = xr.open_mfdataset(file_paths, combine='nested', concat_dim='ensemble')
LENS_PV_mean = LENS_PV_all.PV.isel(time=slice(time_start,time_end)).isel(TEMP = 2).mean(dim='time').compute()
LENS_PV_1deg = proc_utils.regrid_SMYLE(LENS_PV_mean)

regridder = xe.Regridder(LENS_PV_1deg, CESMLENS_var[:,:], 'nearest_s2d', periodic=True)

regridded_LENS_PV = np.absolute(regridder(LENS_PV_1deg))


##### POTENTIAL DENSITY
file_paths = [f'/glade/derecho/scratch/cassiacai/regridded_PD_{member}.nc' for member in ensemble_members]
LENS_PD_all = xr.open_mfdataset(file_paths, combine='nested', concat_dim='ensemble')

ds = LENS_PD_all.PD
regridded_LENS_PD = regrid_time_mean_popvar(ds, regridder, time_start, time_end)
regridded_LENS_PD_nan = xr.where(regridded_LENS_PD == 0., np.nan, regridded_LENS_PD)


In [ ]:
##### ------------  LENS
##### POTENTIAL VORTICITY
file_paths = [f'/glade/derecho/scratch/cassiacai/regridded_PV_{member}.nc' for member in ensemble_members]
LENS_PV_all = xr.open_mfdataset(file_paths, combine='nested', concat_dim='ensemble')
LENS_PV_mean = LENS_PV_all.PV.isel(time=slice(time_start,time_end)).isel(TEMP = 2).mean(dim='time').compute()
LENS_PV_1deg = proc_utils.regrid_SMYLE(LENS_PV_mean)

regridder = xe.Regridder(LENS_PV_1deg, CESMLENS_var[:,:], 'nearest_s2d', periodic=True)

regridded_LENS_PV = np.absolute(regridder(LENS_PV_1deg))


## FOSI data

In [ ]:
##### ------------  FOSI
##### POTENTIAL VORTICITY
FOSI_PV = xr.open_dataset('/glade/derecho/scratch/cassiacai/regridded_PV_FOSI.nc')
FOSI_PV_mean = FOSI_PV.PV.isel(time=slice(time_start,time_end)).isel(TEMP = 2).mean(dim='time').compute()
FOSI_PV_1deg = proc_utils.regrid_SMYLE(FOSI_PV_mean)
regridded_FOSI_PV = np.absolute(regridder(FOSI_PV_1deg))

##### POTENTIAL DENSITY
FOSI_PD = xr.open_dataset('/glade/derecho/scratch/cassiacai/regridded_PD_FOSI_other.nc')
FOSI_PD_mean = FOSI_PD.PD.isel(time=slice(time_start,time_end)).isel(TEMP = 2).mean(dim='time').compute()
FOSI_PD_1deg = proc_utils.regrid_SMYLE(FOSI_PD_mean)
regridded_FOSI_PD = np.absolute(regridder(FOSI_PD_1deg))

##### TEMPERATURE
FOSI_TEMP = xr.open_dataset('/glade/derecho/scratch/cassiacai/regridded_TEMP_FOSI.nc')
FOSI_TEMP_mean = FOSI_TEMP.__xarray_dataarray_variable__.isel(time=slice(time_start,time_end)).isel(TEMP = 2).mean(dim='time').compute()
FOSI_TEMP_1deg = proc_utils.regrid_SMYLE(FOSI_TEMP_mean)
regridded_FOSI_TEMP = np.absolute(regridder(FOSI_TEMP_1deg))

In [ ]:
##### ------------  FOSI
##### POTENTIAL VORTICITY
FOSI_PV = xr.open_dataset('/glade/derecho/scratch/cassiacai/regridded_PV_FOSI.nc')
FOSI_PV_mean = FOSI_PV.PV.isel(time=slice(time_start,time_end)).isel(TEMP = 2).mean(dim='time').compute()
FOSI_PV_1deg = proc_utils.regrid_SMYLE(FOSI_PV_mean)
regridded_FOSI_PV = np.absolute(regridder(FOSI_PV_1deg))

## Potential Density

In [ ]:
regridded_FOSI_PD_nan = xr.where(regridded_FOSI_PD == 0., np.nan, regridded_FOSI_PD)
regridded_FOSI_PD_nan_noAtlantic = xr.where(regridded_FOSI_PD_nan > 25.95, np.nan, regridded_FOSI_PD_nan)

In [ ]:
regridded_LENS_PD_nan = xr.where(regridded_LENS_PD == 0., np.nan, regridded_LENS_PD)
regridded_LENS_PD_nan_noAtlantic = xr.where(regridded_LENS_PD_nan > 25.95, np.nan, regridded_LENS_PD_nan)

In [ ]:
diff = regridded_LENS_PD_nan_noAtlantic.mean(dim='ensemble') - regridded_FOSI_PD_nan_noAtlantic

In [ ]:
# LENS --------- Mean PD on 20°C Isotherm
fig, ax = plt.subplots(figsize=(5, 3), 
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 290, -40, 40], crs=ccrs.PlateCarree())
ax.set_aspect('auto')  # Override the equal aspect ratio

gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 12}  # Longitude labels
gl.ylabel_style = {'size': 12}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='dotted', linewidth=2, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)

regridded_LENS_PD_nan_noAtlantic.mean(dim='ensemble').sel(lat=slice(-40, 40), lon=slice(120,290)).plot.contourf(
    vmin=23, vmax=25.5, levels=11, cmap=cmocean.cm.dense, transform=ccrs.PlateCarree(), add_colorbar=True)

contour_obj = (regridded_LENS_PD_nan_noAtlantic.mean(dim='ensemble')).plot.contour(levels=[23, 23.5, 24, 24.5, 25, 25.5, 26], cmap='k',transform=ccrs.PlateCarree())
(regridded_LENS_PD_nan_noAtlantic.mean(dim='ensemble')).plot.contour(levels=[24.4, 24.5, 24.6], cmap='hotpink', transform=ccrs.PlateCarree())

(regridded_LENS_PD_nan_noAtlantic.mean(dim='ensemble')).plot.contour(levels=[24.9, 25.0, 25.1], cmap='cyan', transform=ccrs.PlateCarree())


plt.title('')
plt.title('LENS: PD on 20°C Isotherm', fontsize=15, fontweight='bold', zorder=12, loc='left')
plt.show()

In [ ]:
# FOSI --------- Mean PD on 20°C Isotherm
fig, ax = plt.subplots(figsize=(5, 3), 
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 290, -40, 40], crs=ccrs.PlateCarree())
ax.set_aspect('auto')  # Override the equal aspect ratio

gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 12}  # Longitude labels
gl.ylabel_style = {'size': 12}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='dotted', linewidth=2, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)

regridded_FOSI_PD_nan_noAtlantic.sel(lat=slice(-40, 40), lon=slice(120,290)).plot.contourf(
    vmin=23, vmax=25.5, levels=11, cmap=cmocean.cm.dense, transform=ccrs.PlateCarree(), add_colorbar=True)

contour_obj = (regridded_FOSI_PD_nan_noAtlantic).plot.contour(levels=[23, 23.5, 24, 24.5, 25, 25.5, 26], cmap='k',transform=ccrs.PlateCarree())
(regridded_FOSI_PD_nan_noAtlantic).plot.contour(levels=[24.4, 24.5, 24.6], cmap='hotpink', transform=ccrs.PlateCarree())

(regridded_FOSI_PD_nan_noAtlantic).plot.contour(levels=[24.9, 25.0, 25.1], cmap='cyan', transform=ccrs.PlateCarree())

plt.title('')
plt.title('FOSI: PD on 20°C Isotherm', fontsize=15, fontweight='bold', zorder=12, loc='left')
plt.show()

In [ ]:
# LENS - FOSI --------- Mean PD on 20°C Isotherm
fig, ax = plt.subplots(figsize=(5, 3), 
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 290, -40, 40], crs=ccrs.PlateCarree())
ax.set_aspect('auto')  # Override the equal aspect ratio

gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 12}  # Longitude labels
gl.ylabel_style = {'size': 12}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='dotted', linewidth=2, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)

diff.sel(lat=slice(-40, 40), lon=slice(120,290)).plot.contourf(vmin=-0.5, vmax=0.5, levels=21,
    cmap=cmocean.cm.balance, transform=ccrs.PlateCarree(), add_colorbar=True)

contour_obj = (diff).plot.contour(levels=[-0.1, 0, 0.1], cmap='k',transform=ccrs.PlateCarree())

plt.title('')
plt.title('LENS - FOSI: PD on 20°C Isotherm', fontsize=15, fontweight='bold', zorder=12, loc='left')
plt.show()

## Potential Density and Depth of 20°C Isotherm on the Equatorial Thermocline

In [ ]:
LENS_TEMP_20_depth = LENS_TEMP_all.__xarray_dataarray_variable__.z_t.isel(TEMP=2).mean(dim='time').compute()

In [ ]:
FOSI_TEMP_20_depth = FOSI_TEMP.__xarray_dataarray_variable__.z_t.isel(TEMP=2).compute()

In [ ]:
plt.figure(figsize=(5, 3))
mean_114 = regridded_FOSI_PD_nan_noAtlantic.sel(lat=0, method='nearest').isel(lon=slice(110, 280))
x_vals = regridded_FOSI_PD_nan_noAtlantic.sel(lat=0, method='nearest').isel(lon=slice(110, 280)).lon
plt.plot(x_vals, mean_114, c='red', label='LENS')
plt.title('Depth of the 20°C isotherm at the EQ')
plt.ylabel('Density (kg/m3)', fontsize=12)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.xlabel('Degrees East', fontsize=12)
plt.grid(c='k', linestyle='dashed', alpha=0.2)
# plt.ylim(24.6, 25.3)
plt.ylim(24.2, 25.3)

plt.show()

In [ ]:
mean_114_fosi = (FOSI_TEMP_20_depth.mean(dim='time').isel(nlat_t=114))[20:]

lons = mean_114_fosi.TLONG.data
lats = mean_114_fosi.TLAT.data
depths = mean_114_fosi.data

valid_mask = ~np.isnan(lons) & ~np.isnan(lats) & ~np.isnan(depths)
lons_clean = lons[valid_mask]
lats_clean = lats[valid_mask]
depths_clean = depths[valid_mask]

max_FOSI_val = mean_114.sel(lon=slice(160, 260)).max().data
min_FOSI_val = mean_114.sel(lon=slice(160, 260)).min().data
print(max_FOSI_val)
print(min_FOSI_val)
print(max_FOSI_val - min_FOSI_val)

In [ ]:
plt.figure(figsize=(5, 3))
mean_114 = regridded_LENS_PD_nan_noAtlantic.sel(lat=0, method='nearest').mean(dim='ensemble').isel(lon=slice(110, 280))
std_114 = regridded_LENS_PD_nan_noAtlantic.sel(lat=0, method='nearest').std(dim='ensemble').isel(lon=slice(110, 280))

x_vals = regridded_LENS_PD_nan_noAtlantic.sel(lat=0, method='nearest').mean(dim='ensemble').isel(lon=slice(110, 280)).lon
plt.plot(x_vals, mean_114, c='blue', label='LENS')

plt.fill_between(x_vals, 
                mean_114 - std_114, 
                mean_114 + std_114, 
                alpha=0.3, color='blue')

plt.title('Depth of the 20°C isotherm at the EQ')
plt.ylabel('Density (kg/m3)', fontsize=12)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.xlabel('Degrees East', fontsize=12)
plt.grid(c='k', linestyle='dashed', alpha=0.2)
plt.ylim(24.2, 25.3)
plt.show()

In [ ]:
mean_114_fosi = (FOSI_TEMP_20_depth.mean(dim='time').isel(nlat_t=114))[20:]

lons = mean_114_fosi.TLONG.data
lats = mean_114_fosi.TLAT.data
depths = mean_114_fosi.data

valid_mask = ~np.isnan(lons) & ~np.isnan(lats) & ~np.isnan(depths)
lons_clean = lons[valid_mask]
lats_clean = lats[valid_mask]
depths_clean = depths[valid_mask]

max_val = mean_114.sel(lon=slice(160, 260)).max().data
min_val = mean_114.sel(lon=slice(160, 260)).min().data
print(max_val)
print(min_val)
print(max_val - min_val)

In [ ]:
plt.figure(figsize=(5, 3))
mean_114 = (LENS_TEMP_20_depth.mean(dim='ensemble').isel(nlat_t=114)/100)[20:]
std_114 = (LENS_TEMP_20_depth.std(dim='ensemble').isel(nlat_t=114)/100)[20:]

mean_114_fosi = (FOSI_TEMP_20_depth.mean(dim='time').isel(nlat_t=114)/100)[20:]
std_114_fosi = (FOSI_TEMP_20_depth.std(dim='time').isel(nlat_t=114)/100)[20:]

x_vals = LENS_TEMP_20_depth.TLONG.isel(nlat_t=114).data[20:]
plt.plot(x_vals, mean_114, c='blue', label='LENS')
x_vals_fosi = LENS_TEMP_20_depth.TLONG.isel(nlat_t=114).data[20:]
plt.plot(x_vals, mean_114_fosi, c='red', label='FOSI')

plt.fill_between(x_vals, 
                mean_114 - std_114, 
                mean_114 + std_114, 
                alpha=0.3, color='blue')

plt.title('Depth of the 20°C isotherm at the EQ')
plt.ylabel('Depth (m)', fontsize=12)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.xlabel('Degrees East)', fontsize=12)
plt.grid(c='k', linestyle='dashed', alpha=0.2)
plt.ylim(200, 25)
plt.show()

## Depth of 20°C Isotherm

In [ ]:
regridded_LENS_TEMP_depth = LENS_TEMP_all.__xarray_dataarray_variable__.z_t.mean(
    dim='ensemble').isel(TEMP=2).compute()

In [ ]:
regridded_FOSI_TEMP_depth = FOSI_TEMP.__xarray_dataarray_variable__.z_t.isel(TEMP=2).compute()

In [ ]:
LENS_mean_TEMP_depth = regridded_LENS_TEMP_depth.mean(dim='time')
LENS_TEMP_depth_1deg = proc_utils.regrid_SMYLE(LENS_mean_TEMP_depth)
regridded_LENS_TEMP_depth = regridder(LENS_TEMP_depth_1deg)

In [ ]:
FOSI_mean_TEMP_depth = regridded_FOSI_TEMP_depth.mean(dim='time')
FOSI_TEMP_depth_1deg = proc_utils.regrid_SMYLE(FOSI_mean_TEMP_depth)
regridded_FOSI_TEMP_depth = regridder(FOSI_TEMP_depth_1deg)

In [ ]:
diff = regridded_LENS_TEMP_depth - regridded_FOSI_TEMP_depth

In [ ]:
# LENS --------- Mean depth of 20°C Isotherm
fig, ax = plt.subplots(figsize=(5, 3), 
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 290, -40, 40], crs=ccrs.PlateCarree())
ax.set_aspect('auto')  # Override the equal aspect ratio

gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 12}  # Longitude labels
gl.ylabel_style = {'size': 12}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='dotted', linewidth=2, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)

(diff.sel(lat=slice(-40, 40), lon=slice(120,290))/100).plot.contourf(
    vmin=-20, vmax=20, levels=41, cmap=cmocean.cm.balance, transform=ccrs.PlateCarree(), add_colorbar=True)

plt.title('')
plt.title('LENS - FOSI: Depth of 20°C Isotherm', fontsize=15, fontweight='bold', zorder=12, loc='left')
plt.show()

In [ ]:
# LENS --------- Mean depth of 20°C Isotherm
fig, ax = plt.subplots(figsize=(5, 3), 
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 290, -40, 40], crs=ccrs.PlateCarree())
ax.set_aspect('auto')  # Override the equal aspect ratio

gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 12}  # Longitude labels
gl.ylabel_style = {'size': 12}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='dotted', linewidth=2, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)

(regridded_LENS_TEMP_depth.sel(lat=slice(-40, 40), lon=slice(120,290))/100).plot.contourf(
    vmin=0, vmax=250, levels=11, cmap=cmocean.cm.deep, transform=ccrs.PlateCarree(), add_colorbar=True)

plt.title('')
plt.title('LENS: Depth of 20°C Isotherm', fontsize=15, fontweight='bold', zorder=12, loc='left')
plt.show()

In [ ]:
# FOSI --------- Mean depth of 20°C Isotherm
fig, ax = plt.subplots(figsize=(5, 3), 
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 290, -40, 40], crs=ccrs.PlateCarree())
ax.set_aspect('auto')

gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 12}
gl.ylabel_style = {'size': 12}
ax.axhline(y=0, color='k', linestyle='dotted', linewidth=2, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)

(regridded_FOSI_TEMP_depth.sel(lat=slice(-40, 40), lon=slice(120,290))/100).plot.contourf(
    vmin=0, vmax=250, levels=11, cmap=cmocean.cm.deep, transform=ccrs.PlateCarree(), add_colorbar=True)

plt.title('')
plt.title('FOSI: Depth of 20°C Isotherm', fontsize=15, fontweight='bold', zorder=12, loc='left')
plt.show()

## Potential vorticity

In [ ]:
# LENS --------- Mean PV on 20°C Isotherm
fig, ax = plt.subplots(figsize=(4, 4), 
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 290, -30, 30], crs=ccrs.PlateCarree())
ax.set_aspect('auto')  # Override the equal aspect ratio

gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 11}  # Longitude labels
gl.ylabel_style = {'size': 11}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='dotted', linewidth=2, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)

regridded_LENS_PV.mean(dim='ensemble').sel(lat=slice(-30, 30), lon=slice(120,290)).plot.contourf(
    vmin=0, vmax=2e-11, levels=11,
    cmap=cmocean.cm.amp, transform=ccrs.PlateCarree(), add_colorbar=False)

pv_sm = plt.cm.ScalarMappable(cmap=cmocean.cm.amp, norm=plt.Normalize(0, 2e-11))
pv_sm.set_array([])
pv_cbar = plt.colorbar(pv_sm, ax=ax, aspect=30, shrink=0.8, pad=0.1, orientation='horizontal')
pv_cbar.set_label('PV (1/s/cm)', fontsize=11)
tick_values = np.arange(0, 2.3e-11, 0.5e-11)
pv_cbar.set_ticks(tick_values)
pv_cbar.ax.tick_params(labelsize=12)

plt.title('')
plt.title('(c) LENS: PV', fontsize=12, fontweight='bold', zorder=12, loc='left')
plt.show()

In [ ]:
# FOSI --------- Mean PV on 20°C Isotherm
fig, ax = plt.subplots(figsize=(4, 4), 
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 290, -30, 30], crs=ccrs.PlateCarree())
ax.set_aspect('auto')  # Override the equal aspect ratio

gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 11}  # Longitude labels
gl.ylabel_style = {'size': 11}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='dotted', linewidth=2, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)

regridded_FOSI_PV.sel(lat=slice(-30, 30), lon=slice(120,290)).plot.contourf(
    vmin=0, vmax=2e-11, levels=11,
    cmap=cmocean.cm.amp, transform=ccrs.PlateCarree(), add_colorbar=False)

pv_sm = plt.cm.ScalarMappable(cmap=cmocean.cm.amp, norm=plt.Normalize(0, 2e-11))
pv_sm.set_array([])
pv_cbar = plt.colorbar(pv_sm, ax=ax, aspect=30, shrink=0.8, pad=0.1, orientation='horizontal')
pv_cbar.set_label('PV (1/s/cm)', fontsize=11)
tick_values = np.arange(0, 2.3e-11, 0.5e-11)
pv_cbar.set_ticks(tick_values)
pv_cbar.ax.tick_params(labelsize=12)
plt.title('')

plt.title('(d) FOSI: PV', fontsize=12, fontweight='bold', zorder=12, loc='left')
plt.show()

In [ ]:
diff_PV = regridded_LENS_PV - regridded_FOSI_PV

In [ ]:
indices = diff_PV.sel(lat=slice(-12, -5), lon=slice(155,240)).mean(dim=('lat','lon')).argsort().data

reordered_list = [ensemble_members[i] for i in indices]
print(reordered_list)
print(ensemble_members)

In [ ]:
ens_mean = diff_PV.mean(dim='ensemble')

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), 
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 290, -30, 30], crs=ccrs.PlateCarree())
ax.set_aspect('auto')  # Override the equal aspect ratio

gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 11}  # Longitude labels
gl.ylabel_style = {'size': 11}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='dotted', linewidth=2, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)

ens_mean.sel(lat=slice(-40, 40), lon=slice(120,290)).plot.contourf(
    vmin=-0.3e-11, vmax=0.3e-11, levels=16,
    cmap=cmocean.cm.balance,transform=ccrs.PlateCarree(), add_colorbar=False)

# ens_mean.sel(lat=slice(-40, 40), lon=slice(120,290)).plot.contour(
#     levels=[0.3e-12, 0.6e-12, 0.9e-12],
#     cmap='k',transform=ccrs.PlateCarree(), add_colorbar=False)

####### COLORBAR
pv_sm = plt.cm.ScalarMappable(cmap=cmocean.cm.balance, norm=plt.Normalize(-0.3e-11, 0.3e-11))
pv_sm.set_array([])
pv_cbar = plt.colorbar(pv_sm, ax=ax, aspect=30, shrink=0.8, pad=0.1, orientation='horizontal')
pv_cbar.set_label('PV (1/s/cm)', fontsize=11)
tick_values = np.arange(-0.3e-11, 0.34e-11, 0.1e-11)
pv_cbar.set_ticks(tick_values)
pv_cbar.ax.tick_params(labelsize=11)
plt.title('')
plt.title('(e) LENS - FOSI: PV', fontsize=12, fontweight='bold', zorder=15, loc='left')
plt.show()

#### FOSI

In [ ]:
firstyear = 1959
lastyear = 2020 #2020
grid = pop_tools.get_grid('POP_gx1v7')
mask = xr.where((grid['REGION_MASK']>0) & (grid['REGION_MASK']<9), 1, np.nan)
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
fosi_montime_vals = [cftime.DatetimeNoLeap(1958+year, 1+month, 15) for year in range(63) for month in range(12)][0:240]

field = 'PD'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
ds_smyle_fosi_pd = xr.open_dataset(fpath+fname)[field].isel(z_t = 0, time=slice(0, 240)).compute()
ds_smyle_fosi_pd['time'] = fosi_montime_vals

field = 'TEMP'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
ds_smyle_fosi_temp = xr.open_dataset(fpath+fname)[field].isel(z_t = 0, time=slice(0, 240)).compute()
ds_smyle_fosi_temp['time'] = fosi_montime_vals

FOSI_PD_1deg = proc_utils.regrid_SMYLE(ds_smyle_fosi_pd.mean(dim='time'))
FOSI_SST_1deg = proc_utils.regrid_SMYLE(ds_smyle_fosi_temp.mean(dim='time'))

FOSI_PD_1deg = FOSI_PD_1deg.where(
    FOSI_PD_1deg!=0, np.nan)
FOSI_SST_1deg = FOSI_SST_1deg.where(
    FOSI_SST_1deg!=0, np.nan)

regridded_FOSI_PD_surf = regridder(FOSI_PD_1deg)
regridded_FOSI_SST_surf = regridder(FOSI_SST_1deg)

In [ ]:
field = 'PV'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
ds_smyle_fosi_PV = xr.open_dataset(fpath+fname)[field].isel(z_t = slice(0, 27), time=slice(0, 240)).compute()
ds_smyle_fosi_PV['time'] = fosi_montime_vals


In [ ]:
# # Calculate seasonal means for DJF (boreal winter)
# FOSI_TAUX_DJF = regridded_fosi_TAUX.where(regridded_fosi_TAUX.time.dt.month.isin([12, 1, 2]), drop=True).mean(dim='time')
# FOSI_TAUY_DJF = regridded_fosi_TAUY.where(regridded_fosi_TAUY.time.dt.month.isin([12, 1, 2]), drop=True).mean(dim='time')

In [ ]:
ENS_MEMB = 0
LENS_PD = ocn_var_ens(ENS_MEMB, 'PD').isel(time=slice(0,240), z_t =0).compute()

In [ ]:
# LENS --- South Pacific
ENS_MEMBERS_LIST = [0, 65,  32, 85, 61, 90, 80, 68, 73, 49] 
INIT_DEPTH = 50

region_indx_list = []; da_list = []
lon_trunc_list = []; lat_trunc_list = []; z_trunc_list = []

for ENS_MEMB in ENS_MEMBERS_LIST:
    file_path = '/glade/derecho/scratch/cassiacai/particle_trajectories/particle_trajectories_lens{}_SH_startingat{}m_1958_1977.zarr'.format(
        ENS_MEMB, INIT_DEPTH)

    lens_pt = xr.open_zarr(file_path)
    l_time, l_lon, l_lat, l_z = compute_data(lens_pt)
    print(ENS_MEMB)
    
    ### SOUTH PACIFIC
    da_l, region_idx_l,lon_trunc_l,lat_trunc_l, z_trunc_l = calc_density(
       lon_data=l_lon, lat_data=l_lat, z_data=l_z, 
        lat_min=-40, lat_max=-30, lon_min=220, lon_max=240,
        timestart=0, timeend=244)
    
    region_indx_list.append(region_idx_l)
    lon_trunc_list.append(lon_trunc_l)
    lat_trunc_list.append(lat_trunc_l)
    z_trunc_list.append(z_trunc_l)
    da_list.append(da_l)

In [ ]:
INIT_DEPTH = 50

file_path = '/glade/derecho/scratch/cassiacai/particle_trajectories/particle_trajectories_fosi_SH_startingat{}m_1958_1977.zarr'.format(
    INIT_DEPTH)

fosi_pt = xr.open_zarr(file_path)
f_time, f_lon, f_lat, f_z = compute_data(fosi_pt)

In [ ]:
firstyear = 1959
lastyear = 2020 #2020
grid = pop_tools.get_grid('POP_gx1v7')
mask = xr.where((grid['REGION_MASK']>0) & (grid['REGION_MASK']<9), 1, np.nan)
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
fosi_montime_vals = [cftime.DatetimeNoLeap(1958+year, 1+month, 15) for year in range(63) for month in range(12)][0:240]

field = 'PD'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
ds_smyle_fosi_pd = xr.open_dataset(fpath+fname)[field].isel(z_t = 0, time=slice(0, 240)).compute()
ds_smyle_fosi_pd['time'] = fosi_montime_vals

FOSI_PD_1deg = proc_utils.regrid_SMYLE(ds_smyle_fosi_pd)#.mean(dim='time'))
FOSI_PD_DJF = FOSI_PD_1deg.where(FOSI_PD_1deg.time.dt.month.isin([6, 7, 8]), drop=True).mean(dim='time')

regridder = xe.Regridder(FOSI_PD_1deg, CESMLENS_var[:,:,:], 'nearest_s2d', periodic=True)

regridded_FOSI_PD_surf = regridder(FOSI_PD_DJF)

In [ ]:
lon_data=f_lon
lat_data=f_lat
z_data=f_z

lon_data, lat_data, z_data= truncate_lonlatz(
    lon_data, lat_data, z_data)

sigma_theta_pd = regridded_FOSI_PD_surf*1000-1000 # regridded_LENS_PD_surf*1000-1000
yellow_indices = find_sigma_theta_points(lon_data, lat_data, sigma_theta_pd, sigma_min=24.4, sigma_max=24.6)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3), 
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -50, 50], crs=ccrs.PlateCarree())
ax.set_aspect('auto')  # Override the equal aspect ratio

gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 12}  # Longitude labels
gl.ylabel_style = {'size': 12}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)

####### CONTOURF
cf = (regridded_FOSI_PD_surf*1000-1000).plot.contourf(vmin=23, vmax=27, levels=17, 
                                                      cmap=cmocean.cm.dense,transform=ccrs.PlateCarree(), add_colorbar=False)
# cf = (regridded_FOSI_PD_45_surf*1000-1000).plot.contourf(vmin=23, vmax=27, levels=12, cmap=cmocean.cm.dense,transform=ccrs.PlateCarree(), add_colorbar=False)

(regridded_FOSI_PD_surf*1000-1000).plot.contour(levels=[24.4, 24.8], cmap='hotpink',transform=ccrs.PlateCarree())

####### INITIAL POINTS
lon_data = lsh_lon_5_p1; lat_data = lsh_lat_5_p1
ax.scatter(lon_data[:, 0], lat_data[:, 0], 
           s=1.5, marker='o',  c='k', norm=plt.Normalize(5000, 18000), edgecolor=None, 
           alpha=1, zorder=100,transform=ccrs.PlateCarree())

# region_idx_f = yellow_indices
# ax.scatter(lon_data[region_idx_f[:], 0], lat_data[region_idx_f[:], 0], 
#            s=1.5, c='cyan', marker='o', edgecolor='cyan', alpha=1, zorder=100,transform=ccrs.PlateCarree())

# ####### TRAJECTORIES
# for i in yellow_indices[::2]:
#     lon = lon_data.isel(trajectory=i).values
#     lat = lat_data.isel(trajectory=i).values
#     depth = z_data.isel(trajectory=i).values
#     points = np.array([lon, lat]).T.reshape(-1, 1, 2)
#     segments = np.concatenate([points[:-1], points[1:]], axis=1)
#     lc = mcollections.LineCollection(
#             segments, cmap='autumn_r', norm=plt.Normalize(0, 30000), 
#             linewidth=0.8, alpha=1, transform=ccrs.PlateCarree())
#     lc.set_array(depth[:-1])
#     ax.add_collection(lc)
    
lon_data = lnh_lon_5_p1; lat_data = lnh_lat_5_p1
ax.scatter(lon_data[:, 0], lat_data[:, 0], 
           s=1.5, c='k', marker='o', edgecolor=None, alpha=1, zorder=100,transform=ccrs.PlateCarree())
region_idx_f = yellow_indices
ax.scatter(lon_data[region_idx_f[:], 0], lat_data[region_idx_f[:], 0], 
           s=1.5, c='cyan', marker='o', edgecolor='cyan', alpha=1, zorder=100,transform=ccrs.PlateCarree())

# ####### COLORBAR
# density_sm = plt.cm.ScalarMappable(cmap=cmocean.cm.dense, norm=plt.Normalize(23, 27))
# density_sm.set_array([])
# density_cbar = plt.colorbar(density_sm, ax=ax, aspect=30, shrink=0.8, pad=0.1, orientation='horizontal')
# density_cbar.set_label('Potential Density (kg/m3)', fontsize=11)
# tick_values = np.arange(23, 28, 1)  # This gives 5, 10, 15, 20, 25, 30
# density_cbar.set_ticks(tick_values)
# density_cbar.ax.tick_params(labelsize=12)
####### TITLE
plt.title(None)
plt.title('(a) FOSI', fontsize=11, fontweight='bold', zorder=12, loc='left')
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

cf = (regridded_LENS_PD_surf*1000-1000).plot.contourf(vmin=23, vmax=27, levels=17, 
                                                      cmap=cmocean.cm.dense)
contour_obj = (regridded_LENS_PD_surf*1000-1000).plot.contour(levels=[23, 23.5, 24, 24.5, 25, 25.5, 26], cmap='k')
(regridded_LENS_PD_surf*1000-1000).plot.contour(levels=[24.4, 24.6, 24.8], cmap='hotpink')

plt.scatter(lsh_lon[:,0], lsh_lat[:,0], s=1, c='k')
# plt.scatter(lshe_lon[:,0], lshe_lat[:,0], s=1, c='k')
plt.scatter(lnh_lon[:,0], lnh_lat[:,0], s=1, c='k')

sigma_theta_pd = regridded_LENS_PD_surf*1000-1000

for i in range(400):  # Assuming you want the first 400 points
    lon = lsh_lon[i, 0]
    lat = lsh_lat[i, 0]
    
    try:
        sigma_val = sigma_theta_pd.sel(lon=lon, lat=lat, method='nearest').values
    except:
        try:
            sigma_val = sigma_theta_pd.interp(lon=lon, lat=lat).values
        except:
            lon_idx = np.argmin(np.abs(sigma_theta_pd.lon.values - lon))
            lat_idx = np.argmin(np.abs(sigma_theta_pd.lat.values - lat))
            sigma_val = sigma_theta_pd.values[lat_idx, lon_idx]
    if 24.4 <= sigma_val <= 24.8:
        color = 'yellow'
    else:
        color = 'k'
    plt.scatter(lon, lat, s=1, c=color, marker='o', zorder=100)

plt.axhline(y=5, c='gray', linestyle='dotted')
plt.axhline(y=0, c='gray', linestyle='dashed')
plt.axhline(y=-5, c='gray', linestyle='dotted')
plt.xlabel('Longitude', fontsize=15)
plt.ylabel('Latitude', fontsize=15)
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.grid(c='k', linestyle='dashed', alpha=0.2)
plt.xlim(110, 305)
plt.ylim(-60, 60)
plt.title('0 LENS: Time-average PD at 5m \nLocations within 24.4-24.8 sigma theta outcrop', fontsize=15)
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

cf = (regridded_FOSI_PD_surf*1000-1000).plot.contourf(vmin=23, vmax=27, levels=17, 
                                                      cmap=cmocean.cm.dense)
contour_obj = (regridded_FOSI_PD_surf*1000-1000).plot.contour(levels=[23, 23.5, 24, 24.5, 25, 25.5, 26], cmap='k')
(regridded_FOSI_PD_surf*1000-1000).plot.contour(levels=[24.4, 24.6, 24.8], cmap='hotpink')
(regridded_FOSI_PD_surf*1000-1000).plot.contour(levels=[25, 25.2, 25.4], cmap='cyan')

# plt.scatter(fsh_lon[:,0], fsh_lat[:,0], s=1, c='k')
# # plt.scatter(lshe_lon[:,0], lshe_lat[:,0], s=1, c='k')
# plt.scatter(fnh_lon[:,0], fnh_lat[:,0], s=1, c='k')

sigma_theta_pd = regridded_FOSI_PD_surf*1000-1000

# for i in range(400):  # Assuming you want the first 400 points
#     lon = fsh_lon[i, 0]
#     lat = fsh_lat[i, 0]
    
#     try:
#         sigma_val = sigma_theta_pd.sel(lon=lon, lat=lat, method='nearest').values
#     except:
#         try:
#             sigma_val = sigma_theta_pd.interp(lon=lon, lat=lat).values
#         except:
#             lon_idx = np.argmin(np.abs(sigma_theta_pd.lon.values - lon))
#             lat_idx = np.argmin(np.abs(sigma_theta_pd.lat.values - lat))
#             sigma_val = sigma_theta_pd.values[lat_idx, lon_idx]
#     if 25 <= sigma_val <= 25.4:
#         color = 'yellow'
#     else:
#         color = 'k'
#     plt.scatter(lon, lat, s=1, c=color, marker='o', zorder=100)

plt.axhline(y=5, c='gray', linestyle='dotted')
plt.axhline(y=0, c='gray', linestyle='dashed')
plt.axhline(y=-5, c='gray', linestyle='dotted')
plt.xlabel('Longitude', fontsize=15)
plt.ylabel('Latitude', fontsize=15)
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.grid(c='k', linestyle='dashed', alpha=0.2)
plt.xlim(110, 305)
plt.ylim(-60, 60)
plt.title('FOSI: Time-average PD at 5m \nLocations within 25.0-25.4 sigma theta outcrop', fontsize=15)
plt.show()

#### Parcel trajectories

In [ ]:
LENS_PD = ocn_var_ens(ENS_MEMB, 'PD').isel(time=slice(0,240), z_t =0).compute()

LENS_PD_1deg = proc_utils.regrid_SMYLE(LENS_PD.mean(dim='time'))

LENS_PD_1deg = FOSI_PD_1deg.where(
    FOSI_PD_1deg!=0, np.nan)

regridded_LENS_PD_surf = regridder(LENS_PD_1deg)

In [ ]:
ENS_MEMBERS_LIST = [0, 65,  32, 85, 61, 90, 80, 68, 73, 49] 
INIT_DEPTH = 50

region_indx_list = []; da_list = []
lon_trunc_list = []; lat_trunc_list = []; z_trunc_list = []

for ENS_MEMB in ENS_MEMBERS_LIST:
    file_path = '/glade/derecho/scratch/cassiacai/particle_trajectories/particle_trajectories_lens{}_SH_startingat{}m_1958_1977.zarr'.format(
        ENS_MEMB, INIT_DEPTH)

    lens_pt = xr.open_zarr(file_path)
    l_time, l_lon, l_lat, l_z = compute_data(lens_pt)
    print(ENS_MEMB)


    LENS_PD = ocn_var_ens(ENS_MEMB, 'PD').isel(time=slice(0,240), z_t =0).compute()
    LENS_PD = ocn_var_ens(ENS_MEMB, 'PD').isel(time=slice(0,240), z_t =0).compute()

    LENS_PD_1deg = proc_utils.regrid_SMYLE(LENS_PD.mean(dim='time'))
    
    LENS_PD_1deg = LENS_PD_1deg.where(
        LENS_PD_1deg!=0, np.nan)
    
    regridded_LENS_PD_surf = regridder(LENS_PD_1deg)

    lon_data=l_lon
    lat_data=l_lat
    z_data=l_z
    
    lon_data, lat_data, z_data= truncate_lonlatz(
        lon_data, lat_data, z_data)
    
    sigma_theta_pd = regridded_LENS_PD_surf*1000-1000 # regridded_LENS_PD_surf*1000-1000
    yellow_indices = find_sigma_theta_points(lon_data, lat_data, sigma_theta_pd, sigma_min=24.4, sigma_max=24.6)

    lat_min = -50
    lat_max = 50
    
    f_xi, f_yi, f_zi = calc_density_sh(lon_data[yellow_indices,:].data.flatten(), lat_data[yellow_indices,:].data.flatten(),gridsize=100, 
                    bw_method=None, lon_bounds=(110, 305), lat_bounds=(-50, 50))
    
    da = xr.DataArray(f_zi, dims=['y', 'x'],
            coords={'lon': (['y', 'x'], f_xi), 'lat': (['y', 'x'], f_yi)}, name='density')
        
    da_selected = da.where((da.lat >= lat_min) & (da.lat <= lat_max), drop=True)

    region_indx_list.append(yellow_indices)
    # lon_trunc_list.append(lon_trunc_l)
    # lat_trunc_list.append(lat_trunc_l)
    # z_trunc_list.append(z_trunc_l)
    da_list.append(da_selected)

In [ ]:
da_xr = xr.concat(da_list, dim='ensemble').mean(dim='ensemble')

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 282, -40, 5], crs=ccrs.PlateCarree())
# ax.set_extent([120, 140, -5, 15], crs=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 12}  # Longitude labels
gl.ylabel_style = {'size': 12}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')  # Override the equal aspect ratio
ax.scatter(lon_data[region_indx_list[-1], 0], lat_data[region_indx_list[-1], 0], 
           s=5, c='hotpink', marker='o', edgecolor='k', zorder=100,transform=ccrs.PlateCarree())
# ax.scatter(lon_data[0][region_idx[0], 0], lat_data[0][region_idx[0], 0], 
#            s=2.5, c='m', marker='o', edgecolor='m', zorder=100,transform=ccrs.PlateCarree())

####### DENSITY CONTOUR
contourf_plot = da_xr.plot.contourf(
    x='lon', y='lat', cmap='Blues', add_colorbar=False, transform=ccrs.PlateCarree(),
    levels=11, 
    vmax=40e-5, 
    vmin=0e-5, alpha=1
)


(regridded_LENS_PD_surf*1000-1000).plot.contour(levels=[24.4, 24.65], cmap='hotpink',transform=ccrs.PlateCarree())

depth_min_m = 500 / 100; depth_max_m = 25000 / 100
sm = plt.cm.ScalarMappable(cmap='jet_r', norm=plt.Normalize(depth_min_m, depth_max_m)); sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, aspect=30, pad=0.1, shrink=0.8,orientation='horizontal')
cbar.ax.tick_params(labelsize=12); cbar.set_label('Depth (m)', fontsize=12)

# TITLE
plt.title('')
plt.title('(d) LENS', fontsize=12, fontweight='bold', zorder=12, loc='left')

plt.show()

In [ ]:
diff = da_xr - da_selected
fig, ax = plt.subplots(figsize=(4, 4),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 282, -40, 5], crs=ccrs.PlateCarree())
# ax.set_extent([120, 140, -5, 15], crs=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 12}  # Longitude labels
gl.ylabel_style = {'size': 12}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')  # Override the equal aspect ratio

# ax.scatter(lon_data[region_idx, 0], lat_data[region_idx, 0], 
#            s=10, c='m', marker='o', edgecolor='m', zorder=100,transform=ccrs.PlateCarree())

####### DENSITY CONTOUR
contourf_plot = diff.plot.contourf(
    x='lon', y='lat', cmap='PRGn', add_colorbar=False, transform=ccrs.PlateCarree(),
    levels=21, 
    vmax=30e-5, 
    vmin=-30e-5, alpha=1
)

diff.plot.contour(
    x='lon', y='lat', transform=ccrs.PlateCarree(), colors='k', linewidths=1,
    levels=[-20e-5, -15e-5, -10e-5, -5e-5, 5e-5, 10e-5, 15e-5, 20e-5], alpha=1)

# (regridded_LENS_PD_surf*1000-1000).plot.contour(levels=[24.4, 24.6], cmap='hotpink',transform=ccrs.PlateCarree())

depth_min_m = 500 / 100; depth_max_m = 25000 / 100
sm = plt.cm.ScalarMappable(cmap='jet_r', norm=plt.Normalize(depth_min_m, depth_max_m)); sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, aspect=30, pad=0.1, shrink=0.8,orientation='horizontal')
cbar.ax.tick_params(labelsize=12); cbar.set_label('Depth (m)', fontsize=12)

# TITLE
plt.title('')
plt.title('(f) LENS - FOSI', fontsize=12, fontweight='bold', zorder=12, loc='left')

plt.show()

In [ ]:

# da = da_f

# f_xi, f_yi, f_zi = calc_density_sh(lsh_lon[yellow_indices,244-61:].data.flatten(), lsh_lat[yellow_indices,244-61:].data.flatten(),gridsize=100, 
#                 bw_method=None, lon_bounds=(110, 305), lat_bounds=(-50, 50))

lon_data=f_lon
lat_data=f_lat
z_data=f_z

region_idx = yellow_indices

lat_min = -50
lat_max = 50

f_xi, f_yi, f_zi = calc_density_sh(lon_data[yellow_indices,:].data.flatten(), lat_data[yellow_indices,:].data.flatten(),gridsize=100, 
                bw_method=None, lon_bounds=(110, 305), lat_bounds=(-50, 50))

# f_xi, f_yi, f_zi = calc_density_sh(lon_data[yellow_indices,:].data.flatten(), lat_data[yellow_indices,:].data.flatten(),
#                 )


da = xr.DataArray(f_zi, dims=['y', 'x'],
        coords={'lon': (['y', 'x'], f_xi), 'lat': (['y', 'x'], f_yi)}, name='density')
    
# da_nan = xr.where(da <= 0.000001, np.nan, da)
da_selected = da.where((da.lat >= lat_min) & (da.lat <= lat_max), drop=True)
# da_selected = da_nan.where((da_nan.lat >= lat_min) & (da_nan.lat <= lat_max), drop=True)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})
####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 282, -40, 5], crs=ccrs.PlateCarree())
# ax.set_extent([120, 140, -5, 15], crs=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 12}  # Longitude labels
gl.ylabel_style = {'size': 12}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')  # Override the equal aspect ratio

# ax.scatter(lon_data[:, 0], lat_data[:, 0], 
#            s=1.5, c='k', marker='o', edgecolor='k', alpha=0.2, zorder=100,transform=ccrs.PlateCarree())

# # ax.scatter(lon_data[yellow_indices, 0], lat_data[yellow_indices, 0], 
# #            s=1.5, c='m', marker='o', edgecolor='m', zorder=100,transform=ccrs.PlateCarree())
ax.scatter(lon_data[region_idx, 0], lat_data[region_idx, 0], 
           s=5, c='hotpink', marker='o', edgecolor='k', zorder=100,transform=ccrs.PlateCarree())

# ax.scatter(lon_data[region_idx[-5:], 0], lat_data[region_idx[-5:], 0], 
#            s=10.5, c='k', marker='o', edgecolor='k', zorder=100,transform=ccrs.PlateCarree())
# ####### TRAJECTORIES
# for i in region_idx[::8]:#region_idx[::8]:
#     lon = lon_data.isel(trajectory=i).values
#     lat = lat_data.isel(trajectory=i).values
#     depth = z_data.isel(trajectory=i).values
#     points = np.array([lon, lat]).T.reshape(-1, 1, 2)
#     segments = np.concatenate([points[:-1], points[1:]], axis=1)
#     lc = mcollections.LineCollection(
#             segments, cmap='jet_r', norm=plt.Normalize(500, 25000), 
#             linewidth=1., alpha=1, transform=ccrs.PlateCarree())
#     lc.set_array(depth[:-1])
#     ax.add_collection(lc)
    
####### DENSITY CONTOUR
contourf_plot = da_selected.plot.contourf(
    x='lon', y='lat', cmap='Blues', add_colorbar=False, transform=ccrs.PlateCarree(),
    levels=11, 
    vmax=40e-5, 
    vmin=0e-5, alpha=1
)

# # FIXED DENSITY COLORBAR WITH BETTER FORMATTING
# density_sm = plt.cm.ScalarMappable(cmap='Blues', 
#                                   norm=plt.Normalize(0.5e-5, 80.5e-5))
# density_sm.set_array([])
# density_cbar = plt.colorbar(density_sm, ax=ax, aspect=30, shrink=0.8, pad=0.1, orientation='horizontal')

# # Cleaner label - explain the units properly
# density_cbar.set_label('Normalized Trajectory Density', fontsize=11)

# # Format ticks to show clean values (0.5, 1.0, 1.5, etc. × 10⁻⁵)
# # tick_values = np.arange(0, 35, 5)  # This gives 5, 10, 15, 20, 25, 30
# tick_values = np.arange(0, 90, 10)  # This gives 5, 10, 15, 20, 25, 30
# density_cbar.set_ticks(tick_values * 1e-5)
# density_cbar.set_ticklabels([f'{x}' for x in tick_values])  # Shows as 5, 10, 15, etc.
# density_cbar.ax.tick_params(labelsize=12)

# # # CONTOUR LINES
# da_selected.plot.contour(
#     x='lon', y='lat', transform=ccrs.PlateCarree(), colors='k', linewidths=1.,
#     levels=[2e-5, 5e-5, 10e-5, 15e-5, 20e-5, 25e-5], alpha=1
# )
(regridded_FOSI_PD_surf*1000-1000).plot.contour(levels=[24.4, 24.6], cmap='hotpink',transform=ccrs.PlateCarree())
# (regridded_FOSI_PD_surf*1000-1000).plot.contour(levels=[24.9, 25.1], alpha=0.5, cmap='cyan',transform=ccrs.PlateCarree())

# DEPTH COLORBAR
depth_min_m = 500 / 100; depth_max_m = 25000 / 100
sm = plt.cm.ScalarMappable(cmap='jet_r', norm=plt.Normalize(depth_min_m, depth_max_m)); sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, aspect=30, pad=0.1, shrink=0.8,orientation='horizontal')
cbar.ax.tick_params(labelsize=12); cbar.set_label('Depth (m)', fontsize=12)

for i in region_idx[::1]:
    lon = lon_data.isel(trajectory=i).values
    lat = lat_data.isel(trajectory=i).values
    depth = z_data.isel(trajectory=i).values
    points = np.array([lon, lat]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    lc = mcollections.LineCollection(
            segments, cmap='Reds', norm=plt.Normalize(0, 25000), 
            linewidth=0.8, alpha=1, transform=ccrs.PlateCarree())
    lc.set_array(depth[:-1])
    ax.add_collection(lc)
    
# TITLE
plt.title('')
plt.title('(e) FOSI', fontsize=12, fontweight='bold', zorder=12, loc='left')

plt.show()

In [ ]:
def regrid_SMYLE(ds, glat=1, glon=1): # from Jacob's notebook
    """
    Inputs:
        ds: xr.DataArray with coordinates that include TLAT and TLONG
    Returns:
        Regridded xr.DataArray with coordinates lat and lon
    """
    ds = ds.rename(({'ULONG': 'lon', 'ULAT': 'lat'}))
    ds_out = xe.util.grid_global(glon, glat)
    regridder = xe.Regridder(ds, ds_out, 'bilinear', periodic=True)
    regridded = regridder(ds)
    new_coords = regridded.assign_coords({'y': regridded.lat[:, 0].values, 'x': regridded.lon[0].values})
    return new_coords.drop_vars(['lat', 'lon']).rename({'x': 'lon', 'y': 'lat'})

## TAUX

In [ ]:
# LENS for regridding purposes
CESMLENS_var = afuncs.LENS_for_regridding() # this is Area

#### FOSI set up
firstyear = 1959
lastyear = 2020
grid = pop_tools.get_grid('POP_gx1v7')
mask = xr.where((grid['REGION_MASK']>0) & (grid['REGION_MASK']<9), 1, np.nan)
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
fosi_montime_vals = [cftime.DatetimeNoLeap(1958+year, 1+month, 15) for year in range(63) for month in range(12)]

In [ ]:
field = 'TEMP'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
ds_smyle_fosi_var = xr.open_dataset(fpath+fname)[field]
ds_smyle_fosi_var['time'] = fosi_montime_vals
var_fosi = ds_smyle_fosi_var.isel(z_t = 0).compute()
fosi_1deg_wzeros_var = regrid_SMYLE(var_fosi)
fosi_1deg_var = fosi_1deg_wzeros_var.where(fosi_1deg_wzeros_var!=0, np.nan)

In [ ]:
field = 'TAUX'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
ds_smyle_fosi_var = xr.open_dataset(fpath+fname)[field]
ds_smyle_fosi_var['time'] = fosi_montime_vals
var_fosi = ds_smyle_fosi_var.isel(time=slice(0, 240)).compute()
fosi_1deg_wzeros_var = regrid_SMYLE(var_fosi)
fosi_1deg_var = fosi_1deg_wzeros_var.where(fosi_1deg_wzeros_var!=0, np.nan)

In [ ]:
%%time
ensemble_members = [0, 65,  32, 85, 61, 90, 80, 68, 73, 49]
regridded_TAUX_lens_list = []
for ENS_MEMB in ensemble_members:
    print(ENS_MEMB)
    TAUX = afuncs.ocn_var_ens(ENS_MEMB, 'TAUX').isel(time=slice(0, 240)).mean(dim='time')
    lens_1deg_wzeros_var = regrid_SMYLE(TAUX)
    lens_1deg_var = lens_1deg_wzeros_var.where(lens_1deg_wzeros_var!=0, np.nan)
    regridder = xe.Regridder(fosi_1deg_var, CESMLENS_var[:,:,:], 'nearest_s2d', periodic=True)
    regridded_lens_TAUX = regridder(lens_1deg_var)
    regridded_TAUX_lens_list.append(regridded_lens_TAUX)

In [ ]:
# ENS_MEMB = 0
# TAUX = afuncs.ocn_var_ens(ENS_MEMB, 'TAUX').isel(time=slice(0, 240)).mean(dim='time')
# lens_1deg_wzeros_var = regrid_SMYLE(TAUX)

In [ ]:
# lens_1deg_var = lens_1deg_wzeros_var.where(lens_1deg_wzeros_var!=0, np.nan)

In [ ]:
regridder = xe.Regridder(fosi_1deg_var, CESMLENS_var[:,:,:], 'nearest_s2d', periodic=True)

In [ ]:
regridded_fosi_TAUX = regridder(fosi_1deg_var)

In [ ]:
regridded_fosi_TAUX.mean(dim='time').plot.contourf()

In [ ]:
regridder = xe.Regridder(fosi_1deg_var, CESMLENS_var[:,:,:], 'nearest_s2d', periodic=True)
regridded_lens_TAUX = regridder(lens_1deg_var)
regridded_fosi_TAUX = regridder(fosi_1deg_var)
diff = regridded_lens_TAUX - regridded_fosi_TAUX.mean(dim='time')

In [ ]:
ensemble_mean_TAUX_lens = xr.concat(regridded_TAUX_lens_list, dim='ensemble').mean(dim='ensemble')

In [ ]:
ensemble_mean_TAUX_lens_comp = ensemble_mean_TAUX_lens.compute()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})

####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
# ax.set_extent([120, 284, -50, 5], crs=ccrs.PlateCarree())
ax.set_extent([120, 284, -40, 40], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

# #### CONTOURF AND CONTOUR
contourf_plot = ensemble_mean_TAUX_lens_comp.plot.contourf(
    x='lon', y='lat', cmap='PRGn', add_colorbar=False, transform=ccrs.PlateCarree(),
    vmin=-1, vmax=1, levels=21, alpha=1)

#### COLORBAR
density_sm = plt.cm.ScalarMappable(cmap='PRGn', norm=plt.Normalize(-10, 10))
density_sm.set_array([])
density_cbar = plt.colorbar(density_sm, ax=ax, aspect=30, shrink=0.8, pad=0.1, orientation='horizontal')
density_cbar.set_label('Zonal wind stress (dyne/cm3 e-1)', fontsize=11)
tick_values = np.arange(-10, 12, 2) 
density_cbar.set_ticks(tick_values)
density_cbar.set_ticklabels([f'{x:.0f}' for x in tick_values])

# density_cbar.set_ticklabels([f'{x}' for x in tick_values]) 
density_cbar.ax.tick_params(labelsize=12)


plt.title('(b) LENS: Zonal Wind Stress', fontsize=12, fontweight='bold', zorder=12, loc='left')
plt.show()

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(4, 0.9))

# Balance colorbar  
density_sm2 = plt.cm.ScalarMappable(cmap='PRGn', norm=plt.Normalize(-1, 1))
cbar2 = plt.colorbar(density_sm2, cax=ax1, orientation='horizontal')
# cbar2.set_label('Transport (Sv/deg longitude)', fontsize=12)
cbar2.set_label('N/m2 (e-1)', fontsize=12)

tick_values2 = np.arange(-1, 1.1, 0.25)  # -0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3
cbar2.set_ticks(tick_values2 * 1)
cbar2.set_ticklabels([f'{x:.01f}' for x in tick_values2])
cbar2.ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})

####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
# ax.set_extent([120, 284, -50, 5], crs=ccrs.PlateCarree())
ax.set_extent([120, 284, -40, 40], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

# #### CONTOURF AND CONTOUR
contourf_plot = regridded_fosi_TAUX.mean(dim='time').plot.contourf(
    x='lon', y='lat', cmap='PRGn', add_colorbar=False, transform=ccrs.PlateCarree(),
    vmin=-1, vmax=1, levels=21, alpha=1)

#### COLORBAR
density_sm = plt.cm.ScalarMappable(cmap='PRGn', norm=plt.Normalize(-10, 10))
density_sm.set_array([])
density_cbar = plt.colorbar(density_sm, ax=ax, aspect=30, shrink=0.8, pad=0.1, orientation='horizontal')
density_cbar.set_label('Zonal wind stress (dyne/cm3 e-1)', fontsize=11)
tick_values = np.arange(-10, 12, 2) 
density_cbar.set_ticks(tick_values)
density_cbar.set_ticklabels([f'{x:.0f}' for x in tick_values])

# density_cbar.set_ticklabels([f'{x}' for x in tick_values]) 
density_cbar.ax.tick_params(labelsize=12)


plt.title('(a) FOSI: Zonal Wind Stress', fontsize=12, fontweight='bold', zorder=12, loc='left')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})

####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -40, 40], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

# #### CONTOURF AND CONTOUR
contourf_plot = diff.plot.contourf(
    x='lon', y='lat', add_colorbar=False, transform=ccrs.PlateCarree(),cmap=cmocean.cm.balance,
    vmin=-1, vmax=1, 
    levels=21, alpha=1)

#### COLORBAR
density_sm = plt.cm.ScalarMappable(cmap=cmocean.cm.balance, norm=plt.Normalize(-10, 10))
density_sm.set_array([])
density_cbar = plt.colorbar(density_sm, ax=ax, aspect=30, shrink=0.8, pad=0.1, orientation='horizontal')
density_cbar.set_label('Zonal wind stress (dyne/cm3 e-1)', fontsize=11)
tick_values = np.arange(-10, 12, 2) 
density_cbar.set_ticks(tick_values)
density_cbar.set_ticklabels([f'{x:.0f}' for x in tick_values])

# density_cbar.set_ticklabels([f'{x}' for x in tick_values]) 
density_cbar.ax.tick_params(labelsize=12)

plt.title('(c) LENS - FOSI: Zonal Wind Stress', fontsize=12, fontweight='bold', zorder=12, loc='left')
plt.show()

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(4, 0.9))

# Balance colorbar  
density_sm2 = plt.cm.ScalarMappable(cmap=cmocean.cm.balance, norm=plt.Normalize(-1, 1))
cbar2 = plt.colorbar(density_sm2, cax=ax1, orientation='horizontal')
# cbar2.set_label('Transport (Sv/deg longitude)', fontsize=12)
cbar2.set_label('N/m2 (e-1)', fontsize=12)

tick_values2 = np.arange(-1, 1.1, 0.25)  # -0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3
cbar2.set_ticks(tick_values2 * 1)
cbar2.set_ticklabels([f'{x:.01f}' for x in tick_values2])
cbar2.ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# diff = regridded_lens_TAUX - regridded_fosi_TAUX
diff = ensemble_mean_TAUX_lens - regridded_fosi_TAUX

In [ ]:
ensemble_mean_TAUX_lens = ensemble_mean_TAUX_lens.compute()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})

####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -40, 40], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

# #### CONTOURF AND CONTOUR
contourf_plot = regridded_fosi_TAUX.mean(dim='time').plot.contourf(
    x='lon', y='lat', cmap='Blues', add_colorbar=False, transform=ccrs.PlateCarree(),
    vmin=0, vmax=0.01, levels=12, alpha=1)

plt.title('(b) Surface Zonal Wind Stress', fontsize=12, fontweight='bold', zorder=12, loc='left')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})

####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -40, 40], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

# #### CONTOURF AND CONTOUR
contourf_plot = diff.mean(dim='time').plot.contourf(
    x='lon', y='lat', cmap='PRGn', add_colorbar=False, transform=ccrs.PlateCarree(),
    vmin=-0.2, vmax=0.2, levels=12, alpha=1)

#### COLORBAR
density_sm = plt.cm.ScalarMappable(cmap='PRGn', norm=plt.Normalize(-0.2, 0.2))
density_sm.set_array([])
density_cbar = plt.colorbar(pv_sm, ax=ax, aspect=50, shrink=1, pad=0.1, orientation='horizontal')
density_cbar.set_label('Zonal Wind Stress (dyne/cm2 e-1)', fontsize=11)
tick_values = np.arange(-2, 2.1, 0.5) 
density_cbar.set_ticks(tick_values* 1e-1)
density_cbar.set_ticklabels([f'{x}' for x in tick_values]) 
density_cbar.ax.tick_params(labelsize=10)

plt.title('(b) Surface Zonal Wind Stress', fontsize=12, fontweight='bold', zorder=12, loc='left')
plt.show()

## Wind Stress Curl

In [ ]:
import numpy as np
import xarray as xr
import xgcm
from matplotlib import pyplot as plt

import pop_tools

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
field = 'TAUX'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
ds_smyle_fosi_var = xr.open_dataset(fpath+fname)
ds_smyle_fosi_var['time'] = fosi_montime_vals
TAUX = ds_smyle_fosi_var[field].isel(time=slice(0, 240)).mean(dim='time')

field = 'TAUY'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
ds_smyle_fosi_var = xr.open_dataset(fpath+fname)
ds_smyle_fosi_var['time'] = fosi_montime_vals
TAUY = ds_smyle_fosi_var[field].isel(time=slice(0, 240)).mean(dim='time')

In [ ]:
# Simpler finite difference approach (less accurate but more robust)
def calculate_curl_simple(taux, tauy, dx, dy):
    """Calculate curl using centered finite differences"""
    # Assuming all arrays are on the same grid
    # Use numpy gradient
    dtauy_dx = np.gradient(tauy, axis=1) / dx
    dtaux_dy = np.gradient(taux, axis=0) / dy
    return dtauy_dx - dtaux_dy

# Convert to numpy arrays
variables_xr = xr.Dataset()
taux_np = TAUX.values
tauy_np = TAUY.values
dx_np = ds_smyle_fosi_var.DXU.values  # or DXT depending on grid
dy_np = ds_smyle_fosi_var.DYU.values  # or DYT depending on grid

# Calculate curl
curl_np = calculate_curl_simple(taux_np, tauy_np, dx_np, dy_np)

# Convert back to xarray
curl_simple = xr.DataArray(
    curl_np,
    coords=TAUX.coords,
    dims=TAUX.dims,
    name='wind_stress_curl_simple'
)

In [ ]:
ensemble_members = [0, 65,  32, 85, 61, 90, 80, 68, 73, 49]
TAUX_lens_list=[]
for ENS_MEMB in ensemble_members:
    print(ENS_MEMB)
    TAUX = afuncs.ocn_var_ens(ENS_MEMB, 'TAUX').isel(time=slice(0, 240)).mean(dim='time')
    TAUX_lens_list.append(TAUX)

TAUY_lens_list=[]
for ENS_MEMB in ensemble_members:
    print(ENS_MEMB)
    TAUY = afuncs.ocn_var_ens(ENS_MEMB, 'TAUY').isel(time=slice(0, 240)).mean(dim='time')
    TAUY_lens_list.append(TAUY)

In [ ]:
lens_TAUX_ds = xr.concat(TAUX_lens_list, dim='ensemble')
lens_TAUY_ds = xr.concat(TAUY_lens_list, dim='ensemble')

In [ ]:
# Simpler finite difference approach (less accurate but more robust)
def calculate_curl_simple(taux, tauy, dx, dy):
    """Calculate curl using centered finite differences"""
    # Assuming all arrays are on the same grid
    # Use numpy gradient
    dtauy_dx = np.gradient(tauy, axis=1) / dx
    dtaux_dy = np.gradient(taux, axis=0) / dy
    return dtauy_dx - dtaux_dy

In [ ]:
ens_indx = 0
TAUX_data = lens_TAUX_ds.isel(ensemble=ens_indx)
TAUY_data = lens_TAUY_ds.isel(ensemble=ens_indx)

In [ ]:
curl_simple_list = []
for ens_indx in range(10):
    print(ens_indx)
    TAUX_data = lens_TAUX_ds.isel(ensemble=ens_indx)
    TAUY_data = lens_TAUY_ds.isel(ensemble=ens_indx)

    # Convert to numpy arrays
    variables_xr = xr.Dataset()
    taux_np = TAUX_data.values
    tauy_np = TAUY_data.values
    dx_np = ds_smyle_fosi_var.DXU.values  # or DXT depending on grid
    dy_np = ds_smyle_fosi_var.DYU.values  # or DYT depending on grid
    
    # Calculate curl
    curl_np = calculate_curl_simple(taux_np, tauy_np, dx_np, dy_np)
    
    # Convert back to xarray
    curl_simple = xr.DataArray(
        curl_np,
        coords=TAUX.coords,
        dims=TAUX.dims,
        name='wind_stress_curl_simple'
    )
    curl_simple_list.append(curl_simple)

In [ ]:
lens_curl_taux_ds = xr.concat(curl_simple_list, dim='ensemble')


In [ ]:
diff_1deg_wzeros_var = regrid_SMYLE(lens_curl_taux_ds.std(dim='ensemble'))
diff_1deg_var = diff_1deg_wzeros_var.where(diff_1deg_wzeros_var!=0, np.nan)
regridder = xe.Regridder(diff_1deg_var, CESMLENS_var[0,:,:], 'nearest_s2d', periodic=True)
regridded_lens_diff = regridder(diff_1deg_var)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})

####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
ax.set_extent([120, 284, -45, 45], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

# #### CONTOURF AND CONTOUR
contourf_plot = regridded_lens_diff.plot.contourf(
    x='lon', y='lat', cmap='bone_r', add_colorbar=False, transform=ccrs.PlateCarree(),
    vmin=0, vmax=1e-9, levels=11, alpha=1)

#### COLORBAR
density_sm = plt.cm.ScalarMappable(cmap='bone_r', norm=plt.Normalize(0, 1.))

density_sm.set_array([])
density_cbar = plt.colorbar(density_sm, ax=ax, aspect=30, shrink=0.8, pad=0.1, orientation='horizontal')
density_cbar.set_label('N/m3 (e-9)', fontsize=11)
tick_values = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
density_cbar.set_ticks(tick_values)
density_cbar.set_ticklabels([f'{x}' for x in tick_values]) 
density_cbar.ax.tick_params(labelsize=10)
plt.title('')
plt.title('(i) LENS: Subduction Rate (St. Dev.)', fontsize=10, fontweight='bold', zorder=12, loc='left')
plt.ylim(-40, 40)
plt.show()

In [ ]:
# Convert to numpy arrays
variables_xr = xr.Dataset()
taux_np = TAUX_data.values
tauy_np = TAUY_data.values
dx_np = ds_smyle_fosi_var.DXU.values  # or DXT depending on grid
dy_np = ds_smyle_fosi_var.DYU.values  # or DYT depending on grid

# Calculate curl
curl_np = calculate_curl_simple(taux_np, tauy_np, dx_np, dy_np)

# Convert back to xarray
curl_simple = xr.DataArray(
    curl_np,
    coords=TAUX.coords,
    dims=TAUX.dims,
    name='wind_stress_curl_simple'
)

In [ ]:
# Simpler finite difference approach (less accurate but more robust)
def calculate_curl_simple(taux, tauy, dx, dy):
    """Calculate curl using centered finite differences"""
    # Assuming all arrays are on the same grid
    # Use numpy gradient
    dtauy_dx = np.gradient(tauy, axis=1) / dx
    dtaux_dy = np.gradient(taux, axis=0) / dy
    return dtauy_dx - dtaux_dy

# Convert to numpy arrays
variables_xr = xr.Dataset()
taux_np = lens_TAUX.values
tauy_np = lens_TAUY.values
dx_np = ds_smyle_fosi_var.DXU.values  # or DXT depending on grid
dy_np = ds_smyle_fosi_var.DYU.values  # or DYT depending on grid

# Calculate curl
curl_np = calculate_curl_simple(taux_np, tauy_np, dx_np, dy_np)

# Convert back to xarray
curl_simple = xr.DataArray(
    curl_np,
    coords=TAUX.coords,
    dims=TAUX.dims,
    name='wind_stress_curl_simple'
)

In [ ]:
# Convert to numpy arrays
variables_xr = xr.Dataset()
taux_np = lens_TAUX.values
tauy_np = lens_TAUY.values
dx_np = ds_smyle_fosi_var.DXU.values  # or DXT depending on grid
dy_np = ds_smyle_fosi_var.DYU.values  # or DYT depending on grid

# Calculate curl
curl_np_lens = calculate_curl_simple(taux_np, tauy_np, dx_np, dy_np)

# Convert back to xarray
curl_np_lens = xr.DataArray(
    curl_np_lens,
    coords=TAUX.coords,
    dims=TAUX.dims,
    name='wind_stress_curl_simple'
)

In [ ]:
lens_TAUX = xr.concat(TAUX_lens_list, dim='ensemble').mean(dim='ensemble')

In [ ]:
lens_TAUY = xr.concat(TAUY_lens_list, dim='ensemble').mean(dim='ensemble')

In [ ]:
# Convert to numpy arrays
variables_xr = xr.Dataset()
taux_np = lens_TAUX.values
tauy_np = lens_TAUY.values
dx_np = ds_smyle_fosi_var.DXU.values  # or DXT depending on grid
dy_np = ds_smyle_fosi_var.DYU.values  # or DYT depending on grid

# Calculate curl
curl_np_lens = calculate_curl_simple(taux_np, tauy_np, dx_np, dy_np)

# Convert back to xarray
curl_np_lens = xr.DataArray(
    curl_np_lens,
    coords=TAUX.coords,
    dims=TAUX.dims,
    name='wind_stress_curl_simple'
)

In [ ]:
# Convert back to xarray
curl_np_lens = xr.DataArray(
    curl_np_lens,
    coords=TAUX.coords,
    dims=TAUX.dims,
    name='wind_stress_curl_simple'
)

In [ ]:
diff = curl_np_lens - curl_simple


In [ ]:
diff = xr.DataArray(
    diff.data,
    coords=TAUX.coords,
    dims=TAUX.dims,
    name='wind_stress_curl_simple'
)

In [ ]:
diff_1deg_wzeros_var = regrid_SMYLE(diff)
diff_1deg_var = diff_1deg_wzeros_var.where(diff_1deg_wzeros_var!=0, np.nan)
regridder = xe.Regridder(diff_1deg_var, CESMLENS_var[0,:,:], 'nearest_s2d', periodic=True)
regridded_lens_diff = regridder(diff_1deg_var)
# regridded_fosi_TAUX = regridder(fosi_1deg_var)
# diff = regridded_lens_TAUX - regridded_fosi_TAUX

In [ ]:
regridded_lens_diff.plot.contourf()

In [ ]:
regridded_lens_TAUX = regridder(lens_1deg_var)
regridded_fosi_TAUX = regridder(fosi_1deg_var)

In [ ]:
diff_1deg_wzeros_var = regrid_SMYLE(curl_np_lens)
diff_1deg_var = diff_1deg_wzeros_var.where(diff_1deg_wzeros_var!=0, np.nan)
regridder = xe.Regridder(diff_1deg_var, CESMLENS_var[0,:,:], 'nearest_s2d', periodic=True)
# regridded_lens_TAUX = regridder(lens_1deg_var)

In [ ]:
regridded_lens_curl_simple = regridder(diff_1deg_var)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})

####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
# ax.set_extent([120, 284, -50, 5], crs=ccrs.PlateCarree())
ax.set_extent([120, 284, -40, 40], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

# #### CONTOURF AND CONTOUR
contourf_plot = regridded_lens_curl_simple.plot.contourf(
    x='lon', y='lat', cmap='PRGn', add_colorbar=False, transform=ccrs.PlateCarree(),
    vmin=-1e-8, vmax=1e-8, levels=21, alpha=1)

#### COLORBAR
density_sm = plt.cm.ScalarMappable(cmap='PRGn', norm=plt.Normalize(-10, 10))
density_sm.set_array([])
density_cbar = plt.colorbar(density_sm, ax=ax, aspect=30, shrink=0.8, pad=0.1, orientation='horizontal')
density_cbar.set_label('Wind stress curl (dyne/cm3 e-9)', fontsize=11)
tick_values = np.arange(-10, 12, 2) 
density_cbar.set_ticks(tick_values)
density_cbar.set_ticklabels([f'{x:.0f}' for x in tick_values])

# density_cbar.set_ticklabels([f'{x}' for x in tick_values]) 
density_cbar.ax.tick_params(labelsize=12)

plt.title('(e) LENS: Wind Stress Curl', fontsize=12, fontweight='bold', zorder=12, loc='left')
plt.show()

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(4, 0.9))

# Balance colorbar  
density_sm2 = plt.cm.ScalarMappable(cmap='PRGn', norm=plt.Normalize(-1, 1))
cbar2 = plt.colorbar(density_sm2, cax=ax1, orientation='horizontal')
# cbar2.set_label('Transport (Sv/deg longitude)', fontsize=12)
cbar2.set_label('N/m3 (e-7)', fontsize=12)

tick_values2 = np.arange(-1, 1.1, 0.25)  # -0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3
cbar2.set_ticks(tick_values2 * 1)
cbar2.set_ticklabels([f'{x:.01f}' for x in tick_values2])
cbar2.ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1) = plt.subplots(1, 1, figsize=(4, 0.9))

# Balance colorbar  
density_sm2 = plt.cm.ScalarMappable(cmap=cmocean.cm.balance, norm=plt.Normalize(-1, 1))
cbar2 = plt.colorbar(density_sm2, cax=ax1, orientation='horizontal')
# cbar2.set_label('Transport (Sv/deg longitude)', fontsize=12)
cbar2.set_label('N/m3 (e-7)', fontsize=12)

tick_values2 = np.arange(-1, 1.1, 0.25)  # -0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3
cbar2.set_ticks(tick_values2 * 1)
cbar2.set_ticklabels([f'{x:.01f}' for x in tick_values2])
cbar2.ax.tick_params(labelsize=12)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 5),
                      subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180, globe=None)})

####### STATICS
ax.add_feature(cfeature.LAND, color='lightgray', zorder=100)
ax.add_feature(cfeature.COASTLINE, linewidth=1., zorder=100)
ax.grid(c='k', linestyle='dashed', alpha=0.2, zorder=4)
# ax.set_extent([120, 284, -50, 5], crs=ccrs.PlateCarree())
ax.set_extent([120, 284, -40, 40], crs=ccrs.PlateCarree())
gl = ax.gridlines(draw_labels={'left': True, 'bottom': True, 'right': False, 'top': False}, 
                  zorder=4, linestyle='--', alpha=0.5)
gl.xlabel_style = {'size': 10, 'color':'k'}  # Longitude labels
gl.ylabel_style = {'size': 10, 'color':'k'}  # Latitude labels
ax.axhline(y=0, color='k', linestyle='-', linewidth=1, zorder=5)
ax.spines['geo'].set_edgecolor('black')
ax.spines['geo'].set_linewidth(1.5)
ax.set_aspect('auto')

# #### CONTOURF AND CONTOUR
contourf_plot = regridded_lens_diff.plot.contourf(
    x='lon', y='lat', cmap=cmocean.cm.balance, add_colorbar=False, transform=ccrs.PlateCarree(),
    vmin=-1e-8, vmax=1e-8, levels=16, alpha=1)

#### COLORBAR
density_sm = plt.cm.ScalarMappable(cmap=cmocean.cm.balance, norm=plt.Normalize(-10, 10))
density_sm.set_array([])
density_cbar = plt.colorbar(density_sm, ax=ax, aspect=30, shrink=0.8, pad=0.1, orientation='horizontal')
density_cbar.set_label('Wind stress curl (dyne/cm3 e-9)', fontsize=11)
tick_values = np.arange(-10, 12, 2) 
density_cbar.set_ticks(tick_values)
density_cbar.set_ticklabels([f'{x:.0f}' for x in tick_values])

# density_cbar.set_ticklabels([f'{x}' for x in tick_values]) 
density_cbar.ax.tick_params(labelsize=12)

plt.title('(f) LENS - FOSI: Wind Stress Curl', fontsize=12, fontweight='bold', zorder=12, loc='left')
plt.show()

### Functions

In [ ]:
def pop_find_lat_ind(loc, LATDAT):
    return np.abs(LATDAT[:, 0].values - loc).argmin()

### Calculation

In [ ]:
CONVERSION_FACTOR = (0.01 ** 3) / 1e6  # m^3/s to Sv 

In [ ]:
comp_val = 'ocn'
var_val='VVEL'
ens_memb_index = 0

directory = f'/glade/campaign/cgd/cesm/CESM2-LE/{comp_val}/proc/tseries/month_1/{var_val}/'

ds_var_hist_var, ds_var_fut_var = cesm2_lens_utils.get_ds_var(
    directory, var=var_val, comp=comp_val, index_hist = ens_memb_index)

In [ ]:
# get the latitudes of -20 to 12
nlat_ind_ls = []
for nlat in range(-50, 50):
    nlat_ind = pop_find_lat_ind(nlat, ds_var_hist_var.ULAT)
    nlat_ind_ls.append(nlat_ind)

In [ ]:
print(nlat_ind_ls)

In [ ]:
dxu_in_cm_lat = ds_var_hist_var.DXU[0, nlat_ind_ls, 120:295].compute()
dz_in_cm = ds_var_hist_var.dz[0,:].isel(z_t = slice(0,27)).compute()

In [ ]:
# field = 'VVEL'
# fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
# fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
# ds_smyle_fosi_var = xr.open_dataset(fpath+fname)
# ds_smyle_fosi_var['time'] = fosi_montime_vals
# VVEL = ds_smyle_fosi_var[field].isel(
#         z_t = slice(0,27),
#         nlon = slice(120,295),
#         nlat = nlat_ind_ls).isel(time=slice(0, 240)).mean(dim='time')

In [ ]:
transport = VVEL[:, :, :] * dxu_in_cm_lat * dz_in_cm  # shape: time x z x lon
surface_integrated_transport = transport.isel(z_t = slice(0, 5)).sum(dim='z_t')
subsurface_integrated_transport = transport.isel(z_t = slice(5, 27)).sum(dim='z_t')
surface_transport_Sv = surface_integrated_transport * CONVERSION_FACTOR  # m³/s → Sv
subsurface_transport_Sv = subsurface_integrated_transport * CONVERSION_FACTOR  # m³/s → Sv
subsurface_transport_Sv_nan = xr.where(subsurface_transport_Sv == 0., np.nan, subsurface_transport_Sv)
surface_transport_Sv_nan = xr.where(surface_transport_Sv == 0., np.nan, surface_transport_Sv)
# filepath = '/glade/derecho/scratch/cassiacai/12032025_fosi_subsurface_transport{}.nc'
# subsurface_transport_Sv_nan.to_netcdf(filepath)
# print(subsurface_transport_Sv_nan)
    
# filepath = '/glade/derecho/scratch/cassiacai/12032025_fosi_surface_transport.nc'
# surface_transport_Sv_nan.to_netcdf(filepath)
# print(surface_transport_Sv_nan)

In [ ]:
filepath = '/glade/derecho/scratch/cassiacai/12032025_fosi_subsurface_transport.nc'
subsurface_transport_Sv_nan.to_netcdf(filepath)
print(filepath)
    
filepath = '/glade/derecho/scratch/cassiacai/12032025_fosi_surface_transport.nc'
surface_transport_Sv_nan.to_netcdf(filepath)
print(filepath)

In [ ]:
for ens_memb_index in [0, 65,  32, 85, 61, 90, 80, 68, 73, 49]:
    print(ens_memb_index)
    directory = f'/glade/campaign/cgd/cesm/CESM2-LE/{comp_val}/proc/tseries/month_1/{var_val}/'

    ds_var_hist_var, ds_var_fut_var = cesm2_lens_utils.get_ds_var(
        directory, var=var_val, comp=comp_val, index_hist = ens_memb_index)

    VVEL_ds = ds_var_hist_var[var_val].sel(
        time=slice('1958-01', '2020-12')).isel(
        z_t = slice(0,27),
        nlon = slice(120,295),
        nlat = nlat_ind_ls).isel(time=slice(0, 240)).mean(dim='time')
    VVEL_ds = VVEL_ds.compute()
    transport = VVEL_ds[:, :, :] * dxu_in_cm_lat * dz_in_cm  # shape: time x z x lon
    surface_integrated_transport = transport.isel(z_t = slice(0, 5)).sum(dim='z_t')
    subsurface_integrated_transport = transport.isel(z_t = slice(5, 27)).sum(dim='z_t')
    surface_transport_Sv = surface_integrated_transport * CONVERSION_FACTOR  # m³/s → Sv
    subsurface_transport_Sv = subsurface_integrated_transport * CONVERSION_FACTOR  # m³/s → Sv
    subsurface_transport_Sv_nan = xr.where(subsurface_transport_Sv == 0., np.nan, subsurface_transport_Sv)
    surface_transport_Sv_nan = xr.where(surface_transport_Sv == 0., np.nan, surface_transport_Sv)
    filepath = '/glade/derecho/scratch/cassiacai/12032025_subsurface_transport_{}.nc'.format(ens_memb_index)
    subsurface_transport_Sv_nan.to_netcdf(filepath)
    print(subsurface_transport_Sv_nan)
    
    filepath = '/glade/derecho/scratch/cassiacai/12032025_surface_transport_{}.nc'.format(ens_memb_index)
    surface_transport_Sv_nan.to_netcdf(filepath)
    print(surface_transport_Sv_nan)

In [ ]:
comp_val='ocn'
ens_memb_index = 0

var_val='VVEL'

VVEL_ds = ds_var_hist_var[var_val].sel(
        time=slice('1958-01', '2020-12')).isel(
        z_t = slice(0,27),
        nlon = slice(120,295),
        nlat = nlat_ind_ls).isel(time=slice(0, 240)).mean(dim='time')

In [ ]:
%%time
VVEL_ds = VVEL_ds.compute()

In [ ]:
dz_in_cm = ds_var_hist_var.dz[0,:].isel(z_t = slice(0,27)).compute()

In [ ]:
transport = VVEL_ds[:, :, :] * dxu_in_cm_lat * dz_in_cm  # shape: time x z x lon

In [ ]:
surface_integrated_transport = transport.isel(z_t = slice(0, 5)).sum(dim='z_t')
subsurface_integrated_transport = transport.isel(z_t = slice(5, 27)).sum(dim='z_t')

In [ ]:
surface_transport_Sv = surface_integrated_transport * CONVERSION_FACTOR  # m³/s → Sv
subsurface_transport_Sv = subsurface_integrated_transport * CONVERSION_FACTOR  # m³/s → Sv


In [ ]:
subsurface_transport_Sv_nan = xr.where(subsurface_transport_Sv == 0., np.nan, subsurface_transport_Sv)
surface_transport_Sv_nan = xr.where(surface_transport_Sv == 0., np.nan, surface_transport_Sv)

In [ ]:
filepath = '/glade/derecho/scratch/cassiacai/12032025_subsurface_transport_{}.nc'.format(ens_memb_index)
# subsurface_transport_Sv_nan.to_netcdf(filepath)

# filepath = '/glade/derecho/scratch/cassiacai/12032025_surface_transport_{}.nc'.format(ens_memb_index)
# surface_transport_Sv_nan.to_netcdf(filepath)

In [ ]:
# Subduction (FOSI)
fosi_t_subduction_r = xr.open_dataset('/glade/derecho/scratch/cassiacai/fosi_total_subduction_rate.nc').__xarray_dataarray_variable__

# Subduction (LENS)

In [ ]:
xr.open_dataset('/glade/derecho/scratch/cassiacai/lens_0_upwelling.nc')

In [ ]:
# LENS upwelling
file_paths = [f'/glade/derecho/scratch/cassiacai/lens_{i}_upwelling.nc' for i in range(100)]

upwelling_branch_ds = xr.open_mfdataset(
    file_paths, 
    combine='nested',
    concat_dim='ens_member').__xarray_dataarray_variable__

upwelling_branch_ds = upwelling_branch_ds.compute()

# Processing upwelling branch
upwelling_branch_ds_ulong = upwelling_branch_ds.assign_coords(
    ULONG=(('nlat', 'nlon'), 
           upwelling_branch_ds['ULONG'].values))

all_ts = []
for lat_ind in range(31):
    ts = upwelling_branch_ds_ulong[:, :, lat_ind, :].dropna(dim='nlon').integrate('ULONG')
    all_ts.append(ts)

lens_upwell_ds = xr.DataArray(
    np.array(all_ts), 
    dims=('lat', 'ens_memb', 'time'), 
    coords={
        'lat': np.linspace(-20, 10, 31), 
        'ens_memb': np.arange(100),
        'time': upwelling_branch_ds.time})

In [ ]:

fosi_surface = xr.open_dataset('/glade/derecho/scratch/cassiacai/fosi_upper_transport_10202025.nc')
fosi_surface_transport = fosi_surface['__xarray_dataarray_variable__']

fosi_upwelling = xr.open_dataset('/glade/derecho/scratch/cassiacai/fosi_upwelling_upper_transport_10202025.nc')
fosi_upwelling_transport = fosi_upwelling['__xarray_dataarray_variable__']


In [ ]:
#### FOSI set up
firstyear = 1959
lastyear = 2020
grid = pop_tools.get_grid('POP_gx1v7')
mask = xr.where((grid['REGION_MASK']>0) & (grid['REGION_MASK']<9), 1, np.nan)
fpath = '/glade/campaign/cesm/development/espwg/SMYLE/initial_conditions/SMYLE-FOSI/ocn/proc/tseries/month_1/'
fosi_montime_vals = [cftime.DatetimeNoLeap(1958+year, 1+month, 15) for year in range(63) for month in range(12)]

In [ ]:
## SCRATCH SURFACE FLOW CALCULATION
# Surface flow
#### VVEL
field = 'VVEL'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'
ds_smyle_fosi_vvel = xr.open_dataset(fpath+fname)[field].isel(z_t = slice(0,5))
ds_smyle_fosi_vvel['time'] = fosi_montime_vals

upper_vvel = ds_smyle_fosi_vvel[:,:,:, 120:295].compute()

field = 'TEMP'
fname = f'g.e22.GOMIPECOIAF_JRA-1p4-2018.TL319_g17.SMYLE.005.pop.h.{field}.030601-036812.nc'

dxu_in_cm_lat = xr.open_dataset(fpath+fname).DXU[nlat_ind_ls, 120:295].compute()
dz_in_cm = xr.open_dataset(fpath+fname).dz.isel(z_t = slice(0,5)).compute()
vvel_subselect = upper_vvel[:,:,nlat_ind_ls, :]

CONVERSION_FACTOR = (0.01 ** 3) / 1e6  # m^3/s to Sv 
transport = vvel_subselect[:, :, :] * dxu_in_cm_lat * dz_in_cm  # shape: time x z x lon
integrated_transport = transport.sum(dim='z_t')  # shape: time x lon
transport_Sv = integrated_transport * CONVERSION_FACTOR  # m³/s → Sv
transport_Sv_nan = xr.where(transport_Sv == 0., np.nan, transport_Sv)

transport_Sv_nan.to_netcdf('/glade/derecho/scratch/cassiacai/fosi_upper_transport.nc')